# 0.0 Introduzione

In questo progetto analizzo un toy dataset di esperienze di viaggio **virtuali** erogate tra il 2021 e il 2023. Il mio obiettivo è capire **che cosa piace davvero alle persone** (categorie di esperienza, piattaforme, lingue, mercati) e che cosa si associa a livelli più alti di **soddisfazione** (misurata in **stelle** da 1 a 5).

In questo notebook andrò ad eseguire una **EDA** (Exploratory Data Analysis) rigorosa: parto dal caricamento e dalla validazione dello schema, controllo la qualità dei dati (duplicati, mancanti, coerenza delle date), costruisco indicatori sintetici e visualizzo le distribuzioni principali. Poi confronto le variabili con le **stelle** per isolare pattern utili al prodotto (ad es. quali categorie o piattaforme tendono a performare meglio).

Infine esploro la **dimensione temporale**: trend della soddisfazione, volumi per livello di stelle, evoluzione della distribuzione di categoria/piattaforma/lingua/mercato nel tempo e una lettura “a griglia” tramite **heatmap** delle visite. Gli insight che ricavo alimentano raccomandazioni operative su **offerta, canali e priorità di sviluppo** per il ciclo successivo.

Ti auguro una buona lettura!

*Stefano Nocco*

-----

# 1.0 Dataset Utilizzato

Lavoro su un dataset di **999** righe e **11** colonne.

**Schema (descrizione campi):**

* `visit_date` *(datetime)* — la data in cui l’utente effettua l’esperienza.
* `user_uuid` *(string)* — identificatore utente (pseudonimo).
* `category` *(string)* — tema dell’esperienza: **luxury&relax**, **sport&adventure**, **spa**, **culture**.
* `platform` *(string)* — dispositivo: **tv**, **pc**, **mobile**.
* `language` *(string)* — lingua dell’app: **it**, **en**, **fr**.
* `country` *(string)* — mercato dichiarato: **it**, **uk**, **fr** (variabile “macro”; la granularità dettagliata è in `country_id`).
* `country_id` *(int/codice)* — codice paese (23 valori distinti).
* `pacchetto_id` *(categoria)* — tipologia di pacchetto (4 valori).
* `hotel_id` *(int)* — identificatore struttura (943 valori unici: forte frammentazione, come atteso per un catalogo ampio).
* `subscription_date` *(datetime)* — data di sottoscrizione dell’utente (range 2020–2022).
* `stars` *(int, 1–5)* — valutazione dell’esperienza.
-----

# 2.0 Setup dell'Ambiente

Imposto l’ambiente di lavoro per garantire riproducibilità, stile visuale coerente e warning chiari.

## 2.1 Librerie Utili

In [ ]:
# Setup
from __future__ import annotations

import warnings
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns
import plotly.io as pio

SEED: int = 42
rng = np.random.default_rng(SEED)

# Pandas
pd.options.mode.copy_on_write = True
pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

# Warnings: mostro solo quelli utili
warnings.filterwarnings("once", category=FutureWarning)
warnings.filterwarnings("ignore")
# Stile grafici
plt.rcParams.update({
    "figure.figsize": (9, 5),
    "axes.grid": True,
    "axes.titlesize": 12,
    "axes.labelsize": 11,
    "legend.fontsize": 10,
})
sns.set_theme(context="notebook", style="whitegrid")

# Plotly: Color Palettes

import plotly.io as pio
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

PALETTES = {
    "metaverse_neon": ["#00F5D4","#00BBF9","#9B5DE5","#F15BB5","#FEE440","#5AF2AE","#845EC2","#2EC4B6"],
    "sunset_tropics": ["#EF476F","#FF9F1C","#FFD166","#06D6A0","#118AB2","#073B4C","#F4A261","#E76F51"],
    "ocean_depths":   ["#134E6F","#1AC8ED","#58D7DB","#70E4EF","#89FCFF","#2541B2","#1768AC","#06AED5"],
    "aurora":         ["#0CCE6B","#2E86AB","#5C3C92","#A23B72","#F18F01","#16DB93","#5531A3","#2D6A4F"],
    "okabe_ito":      ["#0072B2","#E69F00","#009E73","#CC79A7","#D55E00","#56B4E9","#F0E442","#000000"],  # colorblind-safe
}

def use_palette(name: str = "metaverse_neon", dark: bool = True, continuous: str = "Viridis"):
    colorway = PALETTES[name]
    base = "plotly_dark" if dark else "plotly_white"
    pio.templates["__mytheme"] = go.layout.Template(layout=dict(colorway=colorway))
    pio.templates.default = base + "+__mytheme"   # template globale
    px.defaults.template = pio.templates.default  # default per px
    px.defaults.color_discrete_sequence = colorway
    px.defaults.color_continuous_scale = continuous

# Esegui UNA VOLTA a inizio notebook:
use_palette("okabe_ito", dark=False)   # cambia qui per un’altra palette

# Ordini canonici
ORDER_CATEGORY = ["luxury&relax", "sport&adventure", "spa", "culture"]
ORDER_PLATFORM = ["tv", "pc", "mobile"]
ORDER_LANGUAGE = ["it", "en", "fr"]
ORDER_MARKET = ["it", "uk", "fr"]
ORDER_STARS = [1, 2, 3, 4, 5]

# Path progetto
PROJECT_DIR = Path.cwd()
DATA_DIR = PROJECT_DIR / "data"
OUTPUT_DIR = PROJECT_DIR / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)

print("Setup completato ✅")

## 2.2 Loading dei Dati

In [ ]:
# Percorso candidati
DATA_PATH = Path("Travel_data Final Project.csv")

# Colonne attese
EXPECTED_COLS = {
    "visit_date",
    "user_uuid",
    "category",
    "platform",
    "language",
    "country",
    "country_id",
    "pacchetto_id",
    "hotel_id",
    "subscription_date",
    "stars",
}

# Dtypes compatti (pandas 2.x): categorie e interi piccoli dove sensato
dtype_map = {
    "user_uuid": "string",
    "category": "category",
    "platform": "category",
    "language": "category",
    "country": "category",
    "country_id": "Int16",
    "pacchetto_id": "Int8",
    "hotel_id": "Int32",
    "stars": "Int8",
}
parse_dates = ["visit_date", "subscription_date"]

# --- Lettura in df_raw ---
df_raw = pd.read_csv(
    DATA_PATH,
    dtype=dtype_map,
    parse_dates=parse_dates,
    dayfirst=True,        # coerente con formati EU
    encoding="utf-8",
)

# --- Validazioni schema ---
missing = EXPECTED_COLS.difference(df_raw.columns)
extra = set(df_raw.columns).difference(EXPECTED_COLS)
if missing:
    raise ValueError(f"Colonne mancanti: {sorted(missing)}")
if extra:
    print(f"Colonne aggiuntive non previste (le mantengo): {sorted(extra)}")

# --- Controlli qualità dati ---
# 1) Duplicati
dup_count = df_raw.duplicated().sum()
if dup_count > 0:
    print(f"Trovati {dup_count} duplicati. Li rimuovo per sicurezza.")
    df_raw = df_raw.drop_duplicates().reset_index(drop=True)

# 2) Valori mancanti (overview)
na_tot = int(df_raw.isna().sum().sum())
print(f"Valori mancanti totali: {na_tot}")
if na_tot > 0:
    print(df_raw.isna().sum().sort_values(ascending=False).head(10))

# 3) Date: range e coerenza
vmin, vmax = df_raw["visit_date"].min(), df_raw["visit_date"].max()
smin, smax = df_raw["subscription_date"].min(), df_raw["subscription_date"].max()
print(f"visit_date: {vmin.date()} → {vmax.date()}")
print(f"subscription_date: {smin.date()} → {smax.date()}")

# 4) Stelle: dominio 1..5
stars_set = set(df_raw["stars"].dropna().unique().tolist())
allowed_stars = set([1, 2, 3, 4, 5])
unexpected = stars_set.difference(allowed_stars)
if unexpected:
    raise ValueError(f"Valori 'stars' inattesi: {sorted(unexpected)} (attesi: 1..5)")

# 5) Domini noti
def _check_domain(df, name: str, col: str, expected: list[str] | None = None) -> None:
    vals = df[col].dropna().astype(str).unique().tolist()
    if expected is not None:
        miss = set(vals).difference(expected)
        if miss:
            print(f"{name}: trovati valori non previsti: {sorted(miss)}")
    print(f"{name}: {len(vals)} valori → {sorted(vals)}")

_check_domain(df_raw, "Category", "category", expected=["luxury&relax", "sport&adventure", "spa", "culture"])
_check_domain(df_raw, "Platform", "platform", expected=["tv", "pc", "mobile"])
_check_domain(df_raw, "Language", "language", expected=["it", "en", "fr"])
_check_domain(df_raw, "Market (country)", "country", expected=["it", "uk", "fr"])

# 6) Tipi finali e shape
print(f"Shape finale: {df_raw.shape[0]} righe × {df_raw.shape[1]} colonne")
print(df_raw.dtypes)

print("Caricamento e validazioni completati (df_raw pronto)")


In [ ]:
df_raw.head(20)

In [ ]:
df_raw.info()

In [ ]:
df_raw.describe()

Il dataset analizzato comprende 999 esperienze di viaggio virtuale effettuate tra il 2021 e il 2023, offerte da un’agenzia che permette tour immersivi nel metaverso. 23 sono le Destinazioni Uniche (Paesi Visitati nel metaverso). Gli hotel visitati sono, invece, 943.

In totale sono presenti 239 utenti unici, ciascuno con una media di ~4 viaggi virtuali a testa (minimo 1, massimo 10). 

Ogni record contiene informazioni sul viaggio virtuale, tra cui la data di visita, la categoria dell’esperienza (es. luxury&relax, spa, culture, sport&adventure), il paese e la lingua dell’utente, la piattaforma utilizzata (PC, mobile o smart TV), l’ID pacchetto (tipo di pacchetto o tour scelto) e il rating in stelle dell’hotel virtuale visitato. 

Di seguito esamino in dettaglio i grafici esplorativi creati per capire i comportamenti degli utenti e le tendenze chiave, con commenti sui risultati emersi dai dati.

-----

# 3.0 EDA: Esplorazione dei dati

In questa fase voglio farmi un’idea chiara del dataset: grandezze di base, qualità (duplicati/mancanti), distribuzioni delle variabili chiave e primi segnali di relazione con le stelle.

## 3.1 Shape, dtypes e overview

In [ ]:
# Overview base
n_rows, n_cols = df_raw.shape
print(f"Il dataset contiene {n_rows} righe e {n_cols} colonne.")

print("\nTipi di dato (sintesi):")
print(df_raw.dtypes)

print("\nIntervalli temporali principali:")
print(f"- visit_date: {df_raw['visit_date'].min().date()} → {df_raw['visit_date'].max().date()}")
print(f"- subscription_date: {df_raw['subscription_date'].min().date()} → {df_raw['subscription_date'].max().date()}")

print("\nCardinalità utili:")
print(f"- utenti unici: {df_raw['user_uuid'].nunique()}")
print(f"- hotel unici: {df_raw['hotel_id'].nunique()}")
print(f"- country_id distinti: {df_raw['country_id'].nunique()}")

## 3.2 Qualità dati: duplicati e valori mancanti

In [ ]:
# Duplicati e NA
dup_all = df_raw.duplicated().sum()
dup_key = df_raw.duplicated(subset=["user_uuid", "visit_date", "hotel_id"]).sum()

print(f"Duplicati sull'intera riga: {dup_all}")
print(f"Duplicati su chiave (user_uuid, visit_date, hotel_id): {dup_key}")

na_per_col = df_raw.isna().sum().sort_values(ascending=False)
print("\nValori mancanti per colonna (prime 10):")
print(na_per_col.head(10))

# Se volessi rimuovere duplicati (qui non serve), applico:
# df_raw = df_raw.drop_duplicates().reset_index(drop=True)

Cosa concludo:
- Nessun duplicato rilevato, né globale né sulla chiave logica.
- I mancanti sono trascurabili (in questa base non ho criticità sulle colonne principali), quindi non applico imputazioni.

## 3.3 Feature Extraction

In [ ]:
# 3.3 Derivazioni semplici
df_base = df_raw.copy()  # lavoro su una copia per non contaminare la sorgente

# Anno e mese (già usati nel notebook)
df_base["visit_year"] = df_base["visit_date"].dt.year.astype("Int16")
df_base["visit_month"] = df_base["visit_date"].dt.to_period("M").astype("string")

# Maturità utente (giorni tra sottoscrizione e visita)
df_base["user_maturity_days"] = (df_base["visit_date"] - df_base["subscription_date"]).dt.days.astype("Int32")

# --- Nuove colonne "giorni" ---
# 1) Giorno della settimana (0=Lunedì ... 6=Domenica)
df_base["visit_weekday"] = df_base["visit_date"].dt.weekday.astype("Int8")

# 2) Nome giorno in italiano (uso mapping per evitare dipendenze da locale di sistema)
WEEKDAY_IT = {0: "Lunedì", 1: "Martedì", 2: "Mercoledì", 3: "Giovedì", 4: "Venerdì", 5: "Sabato", 6: "Domenica"}
df_base["visit_weekday_name"] = df_base["visit_weekday"].map(WEEKDAY_IT).astype("category")

# 3) Flag weekend (Sabato/Domenica)
df_base["is_weekend"] = df_base["visit_weekday"].isin([5, 6])

# 4) Giorno del mese (1..31)
df_base["visit_day"] = df_base["visit_date"].dt.day.astype("Int8")

print("Feature create: visit_year, visit_month, user_maturity_days, visit_weekday, visit_weekday_name, is_weekend, visit_day")
print(df_base[["visit_date", "visit_weekday", "visit_weekday_name", "is_weekend", "visit_day"]].head(5))

# Piccolo controllo di distribuzione dei giorni della settimana
print("\nDistribuzione visite per giorno della settimana:")
print(df_base["visit_weekday_name"].value_counts())

In [ ]:
df_base

In [ ]:
df_base.info()

## 3.3 Correlazione tra le Features

In questa sezione misuro **relazioni lineari/monotone** tra le variabili disponibili. Ricordiamo sempre che **correlazione ≠ causalità**!

Qui costruisco solo una mappa di indizi per capire **quali dimensioni si muovono assieme** a `stars`.

Approccio minimale, non invasivo:

1. **Numeriche vs numeriche**: calcolo **Pearson** (lineare) e **Spearman** (ranghi).
2. **Categoriche vs `stars`**: faccio **one-hot encoding** leggero e guardo le associazioni con `stars` (Spearman).
3. **Categoriche vs categoriche**: stimo **Cramér’s V** 

In [ ]:
# Seleziono numeriche evitando ID ad alta cardinalità che non hanno senso semantico
num_cols = df_base.select_dtypes(include=["number"]).columns.tolist()
drop_num = {"hotel_id", "country_id"}  # ID/codici: di solito non informativi per correlazioni
num_cols = [c for c in num_cols if c not in drop_num]

# Calcolo matrici
corr_pearson  = df_base[num_cols].corr(method="pearson")
corr_spearman = df_base[num_cols].corr(method="spearman")

# Focus su 'stars'
corr_with_stars_p = corr_pearson["stars"].drop(labels=["stars"]).sort_values(ascending=False)
corr_with_stars_s = corr_spearman["stars"].drop(labels=["stars"]).sort_values(ascending=False)
print("Correlazioni con 'stars' — Pearson (top 10):")
display(corr_with_stars_p.head(10).round(3))
print("\nCorrelazioni con 'stars' — Spearman (top 10):")
display(corr_with_stars_s.head(10).round(3))

Tratto `stars` come **variabile ordinale** (1–5); per questo riporto **Spearman**, più robusto quando non c’è linearità perfetta. I segnali attesi sono **deboli** (perché il dataset è abbastanza equilibrato), ma utili come bussola.

In [ ]:
# 3.4.2 One-hot su categoriche principali e correlazione con 'stars'
cats = ["category", "platform", "language", "country", "pacchetto_id"]
cats = [c for c in cats if c in df_base.columns]

X_dum = pd.get_dummies(df_base[cats].astype(str), drop_first=False, dtype=int)

# Concat con stars e calcolo correlazioni (Spearman) solo contro 'stars'
dum_corr = pd.concat([X_dum, df_base["stars"]], axis=1).corr(method="spearman")["stars"].drop("stars").sort_values()

print("Associazioni con 'stars' (one-hot, Spearman) — bottom 8:")
display(dum_corr.head(8).round(3))
print("\nAssociazioni con 'stars' (one-hot, Spearman) — top 8:")
display(dum_corr.tail(8).round(3))

Un coefficiente positivo indica che la presenza di quel **livello categoriale** tende ad associarsi a **valutazioni più alte**; negativo il contrario. Le magnitudini **non sono grandi** (e va bene così). Prendo nota dei **primi 2–3 segnali** per ciascuna famiglia (categoria/piattaforma/lingua/mercato).

In [ ]:
# 3.4.3 Cramér's V tra categoriche

import numpy as np
import pandas as pd
from scipy.stats import chi2_contingency

def cramers_v(x: pd.Series, y: pd.Series) -> float:
    """Bias-corrected Cramér's V for two categorical vectors."""
    table = pd.crosstab(x, y)
    chi2 = chi2_contingency(table, correction=False)[0]
    n = table.values.sum()
    phi2 = chi2 / n
    r, k = table.shape
    phi2corr = max(0, phi2 - ((k - 1)*(r - 1)) / (n - 1))
    rcorr = r - ((r - 1)**2) / (n - 1)
    kcorr = k - ((k - 1)**2) / (n - 1)
    denom = min((kcorr - 1), (rcorr - 1))
    return np.sqrt(phi2corr / denom) if denom > 0 else 0.0

cat_cols = ["category", "platform", "language", "country"]
cat_cols = [c for c in cat_cols if c in df_base.columns]

# Costruisco una matrice Cramér's V
cv_mat = pd.DataFrame(index=cat_cols, columns=cat_cols, dtype=float)
for i, ci in enumerate(cat_cols):
    for cj in cat_cols[i:]:
        v = cramers_v(df_base[ci].astype(str), df_base[cj].astype(str))
        cv_mat.loc[ci, cj] = v
        cv_mat.loc[cj, ci] = v

print("Cramér's V tra variabili categoriche (0=nessuna associazione, 1=associazione forte):")
display(cv_mat.round(3))

Mi serve per capire se, ad esempio, **country** e **lingua** sono molto co-strutturate (ridondanza informativa).
Ecco, in questo caso, in effetti, c'è una correlazione massima. Lo tengo a mente, anche se è una analisi esplorativa.
Se un paio di categoriche sono **troppo correlate** tra loro, nei modelli futuri eviterei di usarle insieme senza regolarizzazione.

In [ ]:
# 3.4.4 Heatmap delle correlazioni numeriche (quick look)
import seaborn as sns
import matplotlib.pyplot as plt

if "stars" in num_cols:
    fig, ax = plt.subplots(figsize=(7.5, 6))
    sns.heatmap(
        corr_spearman.loc[num_cols, num_cols],
        vmin=-1, vmax=1, center=0,
        square=True, linewidths=.5, annot=False
    )
    ax.set_title("Correlazioni (Spearman) — variabili numeriche")
    plt.tight_layout()
    plt.show()
else:
    print("Salto heatmap: 'stars' non è tra le numeriche selezionate.")

# 4.0 EDA: Grafici

## 4.1 Pie charts di ripartizione delle Features

In questa sezione sintetizzo le distribuzioni delle features con **pie chart**. Mantengo i colori già impostati a livello di notebook. Per chiarezza, su ogni fetta mostro **percentuale** e **conteggio (n)**.

In [ ]:
from typing import Iterable, Optional
import pandas as pd
import plotly.express as px

def _value_counts_with_other(
    s: pd.Series,
    top_k: Optional[int] = None,
    min_share: Optional[float] = 0.01,
    dropna_level: bool = True,
    other_label: str = "Altro",
) -> pd.Series:
    """
    value_counts() con:
    - rispetto dell'ordine per Categorical (sort=False),
    - accorpamento livelli piccoli in 'Altro' usando top_k e/o min_share.
    """
    x = s.dropna() if dropna_level else s

    if pd.api.types.is_categorical_dtype(x):
        x = x.cat.remove_unused_categories()
        counts = x.value_counts(dropna=not dropna_level, sort=False)
    else:
        counts = x.value_counts(dropna=not dropna_level, sort=True)

    n = counts.sum()
    if top_k is not None and top_k > 0 and len(counts) > top_k:
        head = counts.head(top_k)
        tail_sum = counts.iloc[top_k:].sum()
        counts = pd.concat([head, pd.Series({other_label: tail_sum})])

    if min_share is not None and min_share > 0:
        mask_small = counts / n < min_share
        if mask_small.any():
            keep = counts[~mask_small]
            other = counts[mask_small].sum()
            if other > 0:
                counts = pd.concat([keep, pd.Series({other_label: other})])

    # Ordine stabile: per categoriche preservo l'ordine, altrimenti ordinato per conteggio desc
    if not pd.api.types.is_categorical_dtype(s):
        counts = counts.sort_values(ascending=False)
    return counts

def _pie_table(df: pd.DataFrame, feature: str, **kwargs) -> pd.DataFrame:
    counts = _value_counts_with_other(df[feature], **kwargs)
    tbl = (
        counts
        .to_frame("count")
        .assign(share=lambda t: (t["count"] / counts.sum() * 100).round(2))
        .rename_axis(feature)
        .reset_index()
    )
    return tbl

def plotly_pie_for_feature(
    df: pd.DataFrame,
    feature: str,
    top_k: Optional[int] = None,
    min_share: Optional[float] = 0.01,
    dropna_level: bool = True,
    hole: float = 0.35,
):
    if feature not in df.columns:
        print(f"Salto: '{feature}' non presente nel DataFrame.")
        return None

    tbl = _pie_table(df, feature, top_k=top_k, min_share=min_share, dropna_level=dropna_level)

    # Titolo leggibile
    title_map = {
        "category": "Distribuzione per categoria",
        "platform": "Distribuzione per piattaforma",
        "language": "Lingua utilizzata",
        "country": "Mercato (Paese)",
        "stars": "Distribuzione per stelle",
    }
    fig = px.pie(
        tbl,
        names=feature,
        values="count",
        hole=hole,
        title=title_map.get(feature, f"Distribuzione di {feature}"),
    )
    # Etichette con % e n
    fig.update_traces(
        texttemplate="%{percent:.1%} (%{value})",
        textposition="inside",
        hovertemplate="<b>%{label}</b><br>n=%{value}<br>%{percent:.1%}<extra></extra>",
        sort=False  # non altero l'ordine determinato sopra
    )
    fig.update_layout(margin=dict(l=10, r=10, t=60, b=10))
    return fig

def plotly_pies_for_features(
    df: pd.DataFrame,
    features: Iterable[str],
    top_k: Optional[int] = None,
    min_share: Optional[float] = 0.01,
    dropna_level: bool = True,
    hole: float = 0.35,
):
    figs = {}
    for feat in features:
        fig = plotly_pie_for_feature(
            df, feature=feat, top_k=top_k, min_share=min_share, dropna_level=dropna_level, hole=hole
        )
        if fig is not None:
            figs[feat] = fig
            fig.show()
    return figs

### 4.1.1 Distribuzione delle Esperienze per Categoria

In [ ]:
# 4.1.1 Distribuzione delle Esperienze per Categoria
plotly_pie_for_feature(df_base, "category", top_k=None, min_share=0.01, hole=0.35)

In [ ]:
# tabellina d’appoggio
vc = df_base["category"].value_counts(dropna=False)
tbl = (
    vc.to_frame("count")
      .assign(share=lambda t: (t["count"] / len(df_base) * 100).round(2))
      .rename_axis("category")
      .reset_index()
      .sort_values("category")
)
display(tbl)

#### Cosa vediamo nel grafico

Il pubblico che arriva qui cerca soprattutto comfort e tempo di qualità: 
- La categoria **luxury&relax** raccoglie circa il 42% delle visite (419 su 999).
- Subito dietro si muove un blocco consistente di persone che vogliono esperienze attive ma accessibili: **sport&adventure** pesa ~26% (262).
- La componente **spa** non è marginale: ~19% (194) suggerisce che il benessere ha un ruolo stabile, non un capriccio occasionale.
- **Culture** si ferma al ~12% (124): meno volume, ma un bisogno preciso, spesso trainato dall’identità dei luoghi.

Nel loro insieme queste quote raccontano una domanda polarizzata su due promesse: da un lato la rigenerazione (relax, spa), dall’altro l’attività guidata (adventure). È un equilibrio interessante: chi vuole “staccare” completamente trova un’offerta ampia; chi preferisce “muoversi” trova proposte leggere, non estreme. Culture fa da cornice editoriale: numericamente più piccola, ma strategica per dare senso al viaggio e per differenziare il racconto delle destinazioni.

#### Dati chiave

* **luxury\&relax**: 419 (**42,0%**)
* **sport\&adventure**: 262 (**26,2%**)
* **spa**: 194 (**19,4%**)
* **culture**: 124 (**12,4%**)
* Totale visite: **999**

#### Cosa tenere a mente

Queste sono visite (non utenti unici). La forte quota di luxury&relax suggerisce che l’offerta “comfort-first” è il baricentro; spa non è residuale ma strutturale; culture è più di nicchia e va raccontata bene per emergere.

### 4.1.2 Distribuzione delle Visite per Mercato e Lingua

In [ ]:
# 4.1.2 Distribuzione delle Visite per Mercato (country) e Lingua
pie_features = ["language", "country"]  # FIX: 'market' → 'country'
pie_features = [c for c in pie_features if c in df_base.columns]

_ = plotly_pies_for_features(df_base, pie_features, top_k=None, min_share=0.01, hole=0.35)

In [ ]:
# tabellina d’appoggio
vc = df_base["country"].value_counts(dropna=False)
tbl = (
    vc.to_frame("count")
      .assign(share=lambda t: (t["count"] / len(df_base) * 100).round(2))
      .rename_axis("Mercato")
      .reset_index()
      .sort_values("Mercato")
)
display(tbl)

#### Cosa vediamo nel grafico

La distribuzione per **Paese/mercato** e per **Lingua** è allineata e mostra un baricentro italiano, seguito dall’area anglofona e poi dalla francofona:

* **Market (Paese):** **IT** raccoglie circa **il 45,5%** delle visite (455 su 999).
* **Market (Paese):** **UK** pesa **\~35,8%** (358).
* **Market (Paese):** **FR** vale **\~18,6%** (186).
* **Lingua:** **it** copre **\~45,5%** (455), **en** **\~35,8%** (358), **fr** **\~18,6%** (186): Ha gli stessi pesi del market.

In pratica, il progetto parla soprattutto al **pubblico italiano**, con una **seconda colonna anglofona** ben presente e una **nicchia francese** stabile. Il fatto che **market** e **lingua** abbiano la **stessa struttura** suggerisce un’assegnazione coerente dei contenuti: chi arriva dall’IT legge italiano, chi arriva dall’UK legge inglese, ecc. Questo rende **semplici e mirate** le scelte di localizzazione (copy, creatività, assistenza).

#### Dati chiave

* **Market (Paese)** — IT **455** (**45,5%**) · UK **358** (**35,8%**) · FR **186** (**18,6%**)
* **Lingua** — it **455** (**45,5%**) · en **358** (**35,8%**) · fr **186** (**18,6%**)
* Totale visite: **999**
* Finestra dati: **2021–2023**

#### Cosa tenere a mente

La sovrapposizione tra Paese e Lingua indica coerenza di target (es. Italia ↔ it).

#### Attenzione! 
I grafici sopra creati mostrano il totale delle visite. Va considerato, però, che il grafico non mostra la ripartizione per utenti Unici, il che potrebbe offrire un'informazione fuorviante (o meglio, un altro tipo di informazione).

Procedo, quindi, a calcolare gli utenti unici.

In [ ]:
# Attenzione: definizione operativa—attribuisco a ciascun utente il primo valore osservato
users = df_base.sort_values("visit_date").drop_duplicates(subset=["user_uuid"])
for col in ["country", "language"]:
    if col in users:
        vc = users[col].value_counts(dropna=False)
        tbl = (
            vc.to_frame("users")
              .assign(share=lambda t: (t["users"] / len(users) * 100).round(2))
              .rename_axis(col).reset_index()
        )
        print(f"Distribuzione utenti unici per {col}:")
        display(tbl)

### 4.1.3  Distribuzione delle Esperienze per Piattaforma

In [ ]:
plotly_pie_for_feature(df_base, "platform", top_k=None, min_share=0.01, hole=0.35)

In [ ]:
vc = df_base["platform"].value_counts(dropna=False)
display(
    vc.to_frame("count")
      .assign(share=lambda t:(t["count"]/len(df_base)*100).round(2))
      .rename_axis("platform").reset_index()
      .sort_values("platform")
)

#### Cosa vediamo nel grafico

La fruizione delle esperienze ha un chiaro baricentro “da salotto”, con un secondo polo “da scrivania” e un segmento mobile più agile:

* **TV** raccoglie circa **il 44,7%** delle visite (**447** su **999**).
* **PC** pesa **\~33,4%** (**334**).
* **Mobile** vale **\~21,8%** (**218**).

Il canale **TV** domina, segno che una parte consistente del pubblico consuma i contenuti in modo **rilassato e continuativo**. **PC** rappresenta un uso più **intenzionale** (ricerca, confronto), mentre **mobile** potrebbe intercettare momenti **brevi** o in movimento. L’insieme racconta un consumo **ibrido**: scoperta e intrattenimento da TV, valutazione e scelta da PC, consultazione rapida da smartphone.

#### Dati chiave

* **tv**: **447** (**44,7%**)
* **pc**: **334** (**33,4%**)
* **mobile**: **218** (**21,8%**)
* Totale visite: **999**
* Finestra dati: **2021–2023**

#### Cosa tenere a mente
* La ripartizione per device può riflettere anche **disponibilità dell’offerta** e **scelte di distribuzione** (es. contenuti o campagne più presenti su un canale).

Dal punto di vista strategico, conviene continuare a ottimizzare l’esperienza su TV e PC, assicurando la migliore qualità grafica e interattiva su questi dispositivi, senza tuttavia trascurare del tutto il mobile che, pur essendo minoritario, potrebbe crescere con l’aumento della potenza degli smartphone e l’uso di visori mobili.

### 4.1.4 Distribuzione delle Esperienze per Tipologia di Pacchetto

In [ ]:
plotly_pie_for_feature(df_base, "pacchetto_id", top_k=None, min_share=0.01, hole=0.35)

In [ ]:
vc = df_base["pacchetto_id"].value_counts(dropna=False).sort_index()
display(
    vc.to_frame("count")
      .assign(share=lambda t:(t["count"]/len(df_base)*100).round(2))
      .rename_axis("pacchetto_id").reset_index()
)

#### Cosa vediamo nel grafico

La ripartizione delle esperienze per **tipologia di pacchetto** è **molto equilibrata**: i quattro ID si dividono il volume in parti quasi uguali, con uno scarto contenuto tra il più scelto e il meno scelto.

* **Pacchetto 2**: circa **26,3%** (**263** su **999**).
* **Pacchetto 1**: **\~25,9%** (**259**).
* **Pacchetto 0**: **\~25,2%** (**252**).
* **Pacchetto 3**: **\~22,5%** (**225**).

Questo profilo suggerisce un **portafoglio bilanciato**: nessuna proposta domina, tutte intercettano una quota rilevante di pubblico. È una buona base per sperimentare **varianti di comunicazione** o **benefit** senza che i risultati siano trascinati da un’unica offerta.

#### Dati chiave

* ID **2**: **263** (**26,3%**)
* ID **1**: **259** (**25,9%**)
* ID **0**: **252** (**25,2%**)
* ID **3**: **225** (**22,5%**)
* Totale visite: **999**
* Finestra dati: **2021–2023**

#### Cosa tenere a mente

La coda è corta: nessuna tipologia è marginale. Interpreto questa ripartizione come segno che l’offerta è ben segmentata senza sovra-enfasi su un solo pacchetto.

### 4.1.5 Distribuzione delle Valutazioni (Stelle)

In [ ]:
plotly_pie_for_feature(df_base, "stars", top_k=None, min_share=0.01, hole=0.35)

In [ ]:
# (Facoltativo) Ordina le stelle 1→5 anche visivamente
stars_order = pd.Categorical(df_base["stars"], categories=[1,2,3,4,5], ordered=True)
vc = stars_order.value_counts(dropna=False)
display(
    vc.to_frame("count")
      .assign(share=lambda t:(t["count"]/len(df_base)*100).round(2))
      .rename_axis("stars").reset_index()
)

#### Cosa vediamo nel grafico

Le **stelle rappresentano il giudizio dell’utente sull’hotel** visitato, non la categoria ufficiale dell’hotel. Le valutazioni sono distribuite in modo **ampio** su tutte le classi: le **4★** e le **5★** sono leggermente più frequenti, ma senza dominare nettamente.

#### Dati chiave

* **1★**: **193** (**19,32%**)
* **2★**: **189** (**18,92%**)
* **3★**: **199** (**19,92%**)
* **4★**: **210** (**21,02%**)
* **5★**: **208** (**20,82%**)
* **Totale visite**: **999**
* **Finestra dati**: **2021–2023**

#### Cosa tenere a mente

* Le stelle sono una **scala discreta (1–5)**: nei confronti tra gruppi è utile affiancare **mediana** e **IQR**.
* I conteggi sono **per visita**: utenti ricorrenti possono comparire più volte. Per una lettura “per utente” conviene prima **aggregare a livello di `user_uuid`** (es. media o mediana delle sue valutazioni).


In [ ]:
# Attenzione: definizione operativa—attribuisco a ciascun utente il primo valore osservato
users = df_base.sort_values("visit_date").drop_duplicates(subset=["user_uuid"])
for col in ["stars"]:
    if col in users:
        vc = users[col].value_counts(dropna=False)
        tbl = (
            vc.to_frame("users")
              .assign(share=lambda t: (t["users"] / len(users) * 100).round(2))
              .rename_axis(col).reset_index()
        )
        print(f"Distribuzione utenti unici per {col}:")
        display(tbl)

### 4.1.6 Distribuzione in base ad anno, mese e giorno di utilizzo

In [ ]:
# Mese dell'anno (nome IT) aggregato cross-anno
if "visit_month_name_it" not in df_base.columns:
    MONTHS_IT = {1:"Gennaio",2:"Febbraio",3:"Marzo",4:"Aprile",5:"Maggio",6:"Giugno",
                 7:"Luglio",8:"Agosto",9:"Settembre",10:"Ottobre",11:"Novembre",12:"Dicembre"}
    df_base["visit_month_num"] = df_base["visit_date"].dt.month.astype("Int8")
    df_base["visit_month_name_it"] = df_base["visit_month_num"].map(MONTHS_IT).astype("category")
    df_base["visit_month_name_it"] = df_base["visit_month_name_it"].cat.set_categories(
        list(MONTHS_IT.values()), ordered=True
    )

# Giorno della settimana (nome IT): già in 3.3, ma ricreo se assente
if "visit_weekday_name" not in df_base.columns:
    WEEKDAY_IT = {0:"Lunedì",1:"Martedì",2:"Mercoledì",3:"Giovedì",4:"Venerdì",5:"Sabato",6:"Domenica"}
    df_base["visit_weekday"] = df_base["visit_date"].dt.weekday.astype("Int8")
    df_base["visit_weekday_name"] = df_base["visit_weekday"].map(WEEKDAY_IT).astype("category")
    df_base["visit_weekday_name"] = df_base["visit_weekday_name"].cat.set_categories(
        ["Lunedì","Martedì","Mercoledì","Giovedì","Venerdì","Sabato","Domenica"], ordered=True
    )

In [ ]:
plotly_pie_for_feature(df_base, "visit_year", top_k=None, min_share=0.0, hole=0.35)

In [ ]:
plotly_pie_for_feature(df_base, "visit_month_name_it", top_k=None, min_share=0.0, hole=0.35)

In [ ]:
plotly_pie_for_feature(df_base, "visit_weekday_name", top_k=None, min_share=0.0, hole=0.35)

In [ ]:
def _tab(col):
    return (
        df_base[col].value_counts(dropna=False)
          .rename("count")
          .to_frame()
          .assign(pct=lambda t: (t["count"]/len(df_base)*100).round(1))
          .rename_axis(col)
          .reset_index()
    )

display(_tab("visit_year").sort_values("visit_year"))
display(_tab("visit_month_name_it").sort_values("visit_month_name_it"))
display(_tab("visit_weekday_name").sort_values("visit_weekday_name"))

#### Cosa vediamo nel grafico

La distribuzione temporale racconta un **2023 predominante**, una **stagionalità quasi piatta** con leggere creste primaverili/autunnali, e un utilizzo che si concentra **a metà settimana**:

* **Per anno**: 2021 **10,5%**, 2022 **28,8%**, **2023 60,7%**.
* **Per mese**: **Aprile 9,01%** è il valore più alto; **Gennaio 8,91%** e **Settembre 8,91%** seguono da vicino. Poi **Novembre 8,71%**, **Agosto 8,51%**, **Dicembre 8,61%**, **Maggio 8,31%**, **Giugno 8,11%**, **Ottobre 8,01%**. I minimi sono **Febbraio 7,51%** e **Marzo 7,51%**; **Luglio 7,91%** è appena sopra.
* **Per giorno della settimana**: il picco è **Giovedì 16,1%**, seguito da **Lunedì 15,2%** e **Domenica 14,8%**. Poi **Sabato 14,2%**, **Mercoledì 13,9%**, **Venerdì 13,4%**, **Martedì 12,3%**.

Il dataset è **sbilanciato sul 2023**; la stagionalità mensile è **molto uniforme** con piccole onde (primavera/autunno), mentre la fruizione **a metà settimana** (giovedì) è la più intensa.

#### Dati chiave

* **Anni** — 2021: **10,5%** · 2022: **28,8%** · 2023: **60,7%**
* **Mesi** — Gen **8,91%** · Feb **7,51%** · Mar **7,51%** · **Apr 9,01%** · Mag **8,31%** · Giu **8,11%** · Lug **7,91%** · **Ago 8,51%** · **Set 8,91%** · Ott **8,01%** · **Nov 8,71%** · **Dic 8,61%**
* **Giorni** — **Lun 15,2%** · **Mar 12,3%** · **Mer 13,9%** · **Gio 16,1%** · **Ven 13,4%** · **Sab 14,2%** · **Dom 14,8%**

#### Cosa tenere a mente

* Il **peso del 2023 (60,7%)** può riflettere sia **crescita reale** sia **migliore copertura/raccolta dati**; Bisogna stare attenti a non confondere i due effetti.
* Il grafico usa **visite** (non **utenti unici**): chi torna più volte incide di più sulle percentuali.
* Le differenze tra mesi e giorni sono **piccole**: In questo caso è meglio interpretarle come **tendenze** utili per pianificare campagne e contenuti, non come segnali strutturali da soli.

-----

## 4.2 Distribuzioni delle Features in base alle Valutazioni

### 4.2.1 Features Numeriche vs Stelle

In [ ]:
# Utils — histogram binning
from __future__ import annotations
import numpy as np
import pandas as pd
from typing import Iterable

def _nice_bin_edges(
    x: Iterable[float],
    target_bins: int = 30,
    snap_zero: bool = False,
    max_bins: int | None = None,
) -> np.ndarray:
    """
    Calcola bordi di bin 'gradevoli' per istogrammi.

    Parametri
    ---------
    x : Iterable[float]
        Dati numerici (NaN verranno ignorati).
    target_bins : int
        Numero desiderato di bin (usato per derivare lo step 'nice').
    snap_zero : bool
        Se True e lo zero cade nel range dei dati, allinea i bordi in modo
        che 0 sia esattamente un edge del bin.
    max_bins : int | None
        Limite superiore ai bin (se None non pone vincoli aggiuntivi).

    Logica
    ------
    - Caso discreto breve: se tutti interi e range ≤ 100 → bin a passi di 1 (bordi a .5).
    - Caso generale: calcola uno step grezzo = range/target_bins, poi lo 'arrotonda'
      al più vicino schema 1–2–2.5–5–10×10^k. Allinea i bordi a questo passo.
    - Se snap_zero è True e 0 ∈ [min, max], usa 0 come origine dei bordi.

    Ritorna
    -------
    np.ndarray
        Array dei bordi (len = nbins + 1).
    """
    s = pd.Series(x, dtype="float64").dropna()
    n = int(s.size)
    if n == 0:
        return np.array([0.0, 1.0])

    x_min, x_max = float(s.min()), float(s.max())
    if x_min == x_max:
        eps = 0.5 if float(x_min).is_integer() else (abs(x_min) * 1e-3 + 1e-3)
        return np.array([x_min - eps, x_max + eps])

    # Caso discreto breve: bin per interi
    if np.all(np.equal(np.mod(s, 1), 0)) and (x_max - x_min) <= 100:
        return np.arange(x_min - 0.5, x_max + 1.5, 1.0)

    # Numero desiderato minimo
    target_bins = int(max(2, target_bins))

    # Step "grezzo" e arrotondamento a 1-2-2.5-5-10 * 10^k
    rng = x_max - x_min
    raw_step = rng / target_bins
    if raw_step <= 0 or not np.isfinite(raw_step):
        raw_step = 1.0

    pow10 = 10.0 ** np.floor(np.log10(raw_step))
    for m in (1.0, 2.0, 2.5, 5.0, 10.0):
        step = m * pow10
        if step >= raw_step:
            break

    # Allineamento bordi
    if snap_zero and (x_min <= 0.0 <= x_max):
        origin = 0.0
    else:
        origin = np.floor(x_min / step) * step

    # Estendi fino a coprire max
    stop = np.ceil(x_max / step) * step
    edges = np.arange(origin, stop + step * 1.0000001, step)

    # Assicura copertura del min
    while edges[0] > x_min:
        edges = np.insert(edges, 0, edges[0] - step)

    # Vincolo max_bins (se richiesto)
    if max_bins is not None and len(edges) - 1 > max_bins:
        # Ricomputa con step più largo
        factor = np.ceil((len(edges) - 1) / max_bins)
        step *= factor
        if snap_zero and (x_min <= 0.0 <= x_max):
            origin = 0.0
        else:
            origin = np.floor(x_min / step) * step
        stop = np.ceil(x_max / step) * step
        edges = np.arange(origin, stop + step * 1.0000001, step)
        while edges[0] > x_min:
            edges = np.insert(edges, 0, edges[0] - step)

    # Safety: almeno 2 bordi
    if edges.size < 2:
        edges = np.array([x_min, x_max])

    return edges

# --- utility: decide se trattare la feature come "discreta" ---
def _is_discrete_feature(s: pd.Series, max_unique: int = 20) -> bool:
    if pd.api.types.is_categorical_dtype(s):
        return True
    if pd.api.types.is_integer_dtype(s) and s.nunique(dropna=True) <= max_unique:
        return True
    return False

def _ordered_categories(s: pd.Series) -> List:
    """Ordina categorie: se Categorical(ordered) usa quell'ordine, altrimenti ordina per tipo (num→asc, str→alfabetico)."""
    if pd.api.types.is_categorical_dtype(s):
        cat = s.dtype
        levels = list(s.cat.remove_unused_categories().cat.categories)
        if getattr(cat, "ordered", False):
            return levels
        # unordered: ordina in modo "naturale"
        nums = [v for v in levels if isinstance(v, (int, np.integer, float, np.floating))]
        strs = [v for v in levels if isinstance(v, str)]
        return sorted(nums) + sorted(strs)
    # non categorica
    vals = s.dropna().unique().tolist()
    nums = [v for v in vals if isinstance(v, (int, np.integer, float, np.floating))]
    strs = [v for v in vals if isinstance(v, str)]
    return sorted(nums) + sorted(strs)

def plotly_side_by_side_stacked_hists_by_stars(
    df: pd.DataFrame,
    features: Iterable[str],
    bins: int = 30,
    clip_quantiles: Optional[Tuple[float, float]] = (0.01, 0.99),
    max_unique_discrete: int = 20,   # soglia per considerare una numerica "discreta"
) -> None:
    """
    Per ogni feature:
      - Se DISCRETA (pochi valori o Categorical): usa categorie in ascissa (nessun bin superfluo).
      - Se CONTINUA: usa bin "umani" (1-2-5 rule) + clipping opzionale.
    Output: due subplot affiancati (sinistra=100% stacked, destra=counts).
    """
    classes = sorted(df["stars"].dropna().unique())
    classes_label = [f"{int(c)}★" for c in classes]

    # === NUOVO: mappa colori coerente per le classi, basata sul template attivo ===
    try:
        color_seq = list(pio.templates[pio.templates.default].layout.colorway)
        if not color_seq:
            raise ValueError
    except Exception:
        color_seq = px.colors.qualitative.Plotly
    STAR_COLORS = {int(c): color_seq[i % len(color_seq)] for i, c in enumerate(classes)}

    for feature in features:
        s = df[feature]
        if s.dropna().empty:
            print(f"[skip] '{feature}': nessun dato.")
            continue

        is_discrete = _is_discrete_feature(s, max_unique=max_unique_discrete)

        # === CASO DISCRETO: categorie sull'asse X ===
        if is_discrete:
            cats = _ordered_categories(s)
            ct = pd.crosstab(s, df["stars"]).reindex(index=cats, fill_value=0)
            ct_pct = ct.div(ct.sum(axis=1).replace(0, np.nan), axis=0).fillna(0.0) * 100.0

            x_labels = [str(c) for c in ct.index]
            fig = make_subplots(
                rows=1, cols=2,
                subplot_titles=(f"{feature} • 100% stacked", f"{feature} • conteggi"),
                horizontal_spacing=0.12,
            )

            # sinistra: percentuali
            for cls, lab in zip(classes, classes_label):
                y = ct_pct.get(cls, pd.Series([0]*len(x_labels), index=ct_pct.index)).values
                fig.add_trace(
                    go.Bar(
                        x=x_labels, y=y, name=lab,
                        marker_color=STAR_COLORS[int(cls)],  # <-- colore fisso
                        hovertemplate=f"{lab}<br>%{{x}}<br>%{{y:.2f}}%<extra></extra>",
                        showlegend=False,
                    ),
                    row=1, col=1
                )

            # destra: conteggi
            for cls, lab in zip(classes, classes_label):
                y = ct.get(cls, pd.Series([0]*len(x_labels), index=ct.index)).values
                fig.add_trace(
                    go.Bar(
                        x=x_labels, y=y, name=lab,
                        marker_color=STAR_COLORS[int(cls)],  # <-- stesso colore
                        hovertemplate=f"{lab}<br>%{{x}}<br>Conteggio: %{{y}}<extra></extra>",
                        showlegend=True, legendgroup=str(cls),
                    ),
                    row=1, col=2
                )

            fig.update_layout(
                barmode="stack",
                title_text=f"Distribuzione di {feature} per stelle",
                margin=dict(l=20, r=20, t=60, b=40),
            )
            fig.update_yaxes(title_text="Percentuale (%)", range=[0, 100], row=1, col=1)
            fig.update_yaxes(title_text="Conteggio", row=1, col=2)
            # === cambiato: tick verticali anche qui
            fig.update_xaxes(title_text=feature, tickangle=90, row=1, col=1)
            fig.update_xaxes(title_text=feature, tickangle=90, row=1, col=2)

            fig.show()
            continue

        # === CASO CONTINUO: bin "umani" + clipping opzionale ===
        series_all = pd.to_numeric(s, errors="coerce").dropna().astype(float)
        if series_all.empty:
            print(f"[skip] '{feature}': nessun dato numerico valido.")
            continue

        if clip_quantiles is not None:
            q_low, q_high = series_all.quantile(clip_quantiles[0]), series_all.quantile(clip_quantiles[1])
            series_all = series_all.clip(q_low, q_high)

        bin_edges = _nice_bin_edges(series_all, target_bins=bins, snap_zero=True)

        def _fmt_edge(v: float) -> str:
            return f"{v:.0f}" if abs(v) >= 100 else (f"{v:.1f}" if abs(v) >= 10 else f"{v:.2f}")
        bin_labels = [f"[{_fmt_edge(l)}, {_fmt_edge(r)})" for l, r in zip(bin_edges[:-1], bin_edges[1:])]

        class_counts = []
        for cls in classes:
            vals = pd.to_numeric(df.loc[df["stars"] == cls, feature], errors="coerce").dropna().astype(float)
            if clip_quantiles is not None and not series_all.empty:
                vals = vals.clip(series_all.min(), series_all.max())
            counts, _ = np.histogram(vals, bins=bin_edges)
            class_counts.append(counts.astype(float))
        class_counts = np.array(class_counts)

        totals = class_counts.sum(axis=0)
        with np.errstate(divide="ignore", invalid="ignore"):
            class_perc = np.where(totals > 0, class_counts / totals * 100.0, 0.0)

        fig = make_subplots(
            rows=1, cols=2,
            subplot_titles=(f"{feature} • 100% stacked", f"{feature} • conteggi"),
            horizontal_spacing=0.12,
        )

        # percentuali
        for i, lab in enumerate(classes_label):
            fig.add_trace(
                go.Bar(
                    x=bin_labels, y=class_perc[i],
                    name=lab,
                    marker_color=STAR_COLORS[int(classes[i])],  # <-- colore fisso
                    hovertemplate=f"{lab}<br>Bin: %{{x}}<br>Percentuale: %{{y:.2f}}%<extra></extra>",
                    showlegend=False,
                ),
                row=1, col=1
            )
        # conteggi
        for i, lab in enumerate(classes_label):
            fig.add_trace(
                go.Bar(
                    x=bin_labels, y=class_counts[i],
                    name=lab,
                    marker_color=STAR_COLORS[int(classes[i])],  # <-- stesso colore
                    hovertemplate=f"{lab}<br>Bin: %{{x}}<br>Conteggio: %{{y}}<extra></extra>",
                    showlegend=True, legendgroup=str(i),
                ),
                row=1, col=2
            )

        fig.update_layout(
            barmode="stack",
            title_text=f"Distribuzione di {feature} per stelle",
            margin=dict(l=20, r=20, t=60, b=40),
        )
        fig.update_yaxes(title_text="Percentuale (%)", range=[0, 100], row=1, col=1)
        fig.update_yaxes(title_text="Conteggio", row=1, col=2)
        fig.update_xaxes(title_text=feature, tickangle=90, row=1, col=1)
        fig.update_xaxes(title_text=feature, tickangle=90, row=1, col=2)

        fig.show()

In [ ]:
# Selezione automatica delle numeriche; i discreti (pochi valori) verranno gestiti come categorie
num_cols = df_base.select_dtypes(include=[np.number]).columns.tolist()
drop_cols = {"stars", "hotel_id", "destination_id"}  # escludi ID
num_features = [c for c in num_cols if c not in drop_cols]

# Esecuzione
plotly_side_by_side_stacked_hists_by_stars(
    df_base,
    features=num_features,  # se pacchetto_id è int, lo include già
    bins=30,                      # target per continui; discreti ignorano questo parametro
    clip_quantiles=(0.01, 0.99),  # clipping outlier per continui
    max_unique_discrete=20,       # soglia per trattare una numerica come discreta
)

#### Pacchetto ID
Le percentuali per pacchetto sono **vicine tra loro**. Si nota **pacchetto 2 un po’ più ricco di 5★** e **pacchetti 0–1 leggermente più carichi di 1★–2★**; 3★ e 4★ restano abbastanza stabili. Nei conteggi, **1 e 2 sono i più popolari**, **3 è il meno numeroso**.

#### Anno della visita
La composizione percentuale delle stelle è simile tra gli anni, con una **lieve crescita della quota 5★ nel 2022–2023** e cali/modesti ribilanciamenti tra 2★–3★. I **volumi aumentano nettamente nel tempo**: campione ridotto nel 2021, maggiore nel 2022, **molto alto nel 2023**.

#### Giorno della settimana
Distribuzioni **molto simili** tra i giorni. Si intravede **leggera crescita delle 4★ a fine settimana** e **1★ un po’ più alta all’inizio/fine**. I conteggi non sono uniformi: **picchi a metà settimana**, **minimi sul giorno 1**.

#### Mese
Composizione percentuale **quasi piatta** lungo l’anno. Indizi deboli: **4★ leggermente più presenti in estate**, **1★ un po’ più visibili a dicembre**. Nei conteggi, **febbraio è il mese più scarno**, mentre **primavera–estate e dicembre** hanno volumi più alti.

#### Anzianità degli iscritti
Sulle percentuali emergono oscillazioni: **la quota di 1★ tende ad aumentare verso le code più “anziane”**, mentre 3★-4★ si comprimono leggermente; tuttavia i **conteggi calano molto dopo ~600 giorni**, quindi **le code hanno bassa affidabilità**. Il grosso delle osservazioni è **tra ~200 e ~400 giorni**.

-----

### 4.2.2 Features Categoriche vs Stelle

In [ ]:
from typing import Iterable, Optional
import numpy as np
import pandas as pd
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import plotly.io as pio
import plotly.express as px  # per fallback palette

def _natural_order(vals: List[Union[str, int, float]]) -> List[Union[str, int, float]]:
    """Ordine 'naturale': numeri in ordine crescente seguiti da stringhe alfabetiche."""
    nums = [v for v in vals if isinstance(v, (int, np.integer, float, np.floating))]
    strs = [v for v in vals if isinstance(v, str)]
    return sorted(nums) + sorted(strs)

def _prep_categorical_series(
    s: pd.Series,
    top_k: Optional[int] = None,
    min_share: Optional[float] = None,
    dropna_level: bool = True,
    other_label: str = "Altro",
) -> pd.Series:
    """
    Applica aggregazione in 'Altro' (top_k/min_share), preservando l'ordine:
    - Se s è Categorical(ordered), mantiene quell'ordine e mette 'Altro' in coda.
    - Altrimenti usa un ordine 'naturale'.
    Ritorna una Series Categorical(ordered=True).
    """
    sr = s.copy()
    if dropna_level:
        sr = sr.dropna()
    if sr.empty:
        return sr

    # Conteggi complessivi (per decidere top_k / min_share)
    counts = sr.value_counts()
    keep_levels = set(counts.index)

    if top_k is not None and top_k > 0 and len(keep_levels) > top_k:
        keep_levels = set(counts.nlargest(top_k).index)

    if min_share is not None and 0 < min_share < 1:
        shares = counts / counts.sum()
        small = set(shares[shares < min_share].index)
        keep_levels = keep_levels - small

    # Mappa i livelli esclusi in 'Altro' (se necessario)
    if len(keep_levels) < counts.size:
        sr = sr.where(sr.isin(list(keep_levels)), other_label)

    # Definisci l'ordine delle categorie
    if pd.api.types.is_categorical_dtype(s) and getattr(s.dtype, "ordered", False):
        base_order = [lvl for lvl in s.cat.categories if lvl in sr.unique()]
    else:
        base_order = _natural_order(list(pd.unique(sr)))

    # 'Altro' sempre in coda
    if other_label in sr.values and other_label not in base_order:
        base_order = [lvl for lvl in base_order if lvl != other_label] + [other_label]
    elif other_label in base_order:
        base_order = [lvl for lvl in base_order if lvl != other_label] + [other_label]

    # Ritorna categorica ordinata, senza livelli non usati
    sr = pd.Categorical(sr, categories=base_order, ordered=True)
    return pd.Series(sr, index=s.index, name=s.name)

def plotly_categorical_side_by_side_vs_stars(
    df: pd.DataFrame,
    features: Iterable[str],
    top_k: Optional[int] = None,       # es. 10 per country
    min_share: Optional[float] = 0.01, # raggruppa sotto l'1% in "Altro"
    drop_na_level: bool = True,        # rimuove righe con feat/stars NaN
    tickangle_x: int = 90,             # etichette asse X verticali
) -> None:
    if "stars" not in df.columns:
        raise ValueError("'stars' non presente nel DataFrame.")
    classes = sorted(df["stars"].dropna().unique())
    classes_label = [f"{int(c)}★" for c in classes]

    # === NUOVO: colori coerenti per le classi in entrambi i pannelli ===
    try:
        color_seq = list(pio.templates[pio.templates.default].layout.colorway)
        if not color_seq:
            raise ValueError
    except Exception:
        color_seq = px.colors.qualitative.Plotly
    STAR_COLORS = {int(c): color_seq[i % len(color_seq)] for i, c in enumerate(classes)}

    for feat in features:
        if feat not in df.columns:
            print(f"[skip] '{feat}' non trovato nel DataFrame.")
            continue

        data = df[[feat, "stars"]].copy()
        if drop_na_level:
            data = data.dropna(subset=["stars", feat])
        else:
            data = data.dropna(subset=["stars"])
        if data.empty:
            print(f"[skip] '{feat}': nessun dato utile.")
            continue

        # Prepara la categoriale (ordina + 'Altro')
        s_cat = _prep_categorical_series(
            data[feat],
            top_k=top_k,
            min_share=min_share,
            dropna_level=drop_na_level,
            other_label="Altro",
        )
        data = data.loc[s_cat.index].copy()
        data[feat] = s_cat

        # Ordine esplicito righe/colonne
        idx_order = list(s_cat.cat.categories)
        ct = pd.crosstab(data[feat], data["stars"])
        ct = ct.reindex(index=idx_order, columns=classes, fill_value=0)

        if ct.empty or ct.sum().sum() == 0:
            print(f"[skip] '{feat}': nessun conteggio.")
            continue

        # 100% per riga (distribuzione stelle per categoria)
        row_sums = ct.sum(axis=1).replace(0, np.nan)
        ct_pct = (ct.div(row_sums, axis=0) * 100.0).fillna(0.0)

        x_labels = [str(x) for x in ct.index.tolist()]
        fig = make_subplots(
            rows=1, cols=2,
            subplot_titles=(f"{feat} • 100% stacked", f"{feat} • conteggi"),
            horizontal_spacing=0.12,
        )

        # Sinistra: percentuali
        for cls, lab in zip(classes, classes_label):
            y_vals = ct_pct.get(cls, pd.Series([0]*len(x_labels), index=ct_pct.index)).values
            fig.add_trace(
                go.Bar(
                    x=x_labels, y=y_vals, name=lab,
                    marker_color=STAR_COLORS[int(cls)],  # <-- colore fisso
                    hovertemplate=f"{lab}<br>%{{x}}<br>Percentuale: %{{y:.2f}}%<extra></extra>",
                    showlegend=False,
                ),
                row=1, col=1
            )

        # Destra: conteggi
        for cls, lab in zip(classes, classes_label):
            y_vals = ct.get(cls, pd.Series([0]*len(x_labels), index=ct.index)).values
            fig.add_trace(
                go.Bar(
                    x=x_labels, y=y_vals, name=lab,
                    marker_color=STAR_COLORS[int(cls)],  # <-- stesso colore
                    hovertemplate=f"{lab}<br>%{{x}}<br>Conteggio: %{{y}}<extra></extra>",
                    showlegend=True, legendgroup=str(cls),
                ),
                row=1, col=2
            )

        fig.update_layout(
            barmode="stack",
            title_text=f"Distribuzione di {feat} per stelle",
            margin=dict(l=20, r=20, t=60, b=40),
        )
        fig.update_yaxes(title_text="Percentuale (%)", range=[0, 100], row=1, col=1)
        fig.update_yaxes(title_text="Conteggio", row=1, col=2)
        fig.update_xaxes(title_text=feat, tickangle=tickangle_x, row=1, col=1)
        fig.update_xaxes(title_text=feat, tickangle=tickangle_x, row=1, col=2)

        fig.show()

In [ ]:
# ===== Esecuzione sulle categoriche principali =====
cat_features = ["category", "language", "country", "platform"]
cat_features = [c for c in cat_features if c in df_base.columns]

# Usa top_k/min_share se hai molte modalità (es. country molto vario)
plotly_categorical_side_by_side_vs_stars(
    df_base,
    features=cat_features,
    top_k=None,          # es. 12 per limitare le categorie più frequenti
    min_share=0.01,      # raggruppa sotto 1% in "Altro" (None per disattivare)
    drop_na_level=True,
    tickangle_x=90       # etichette verticali
)

#### Category
Distribuzioni percentuali molto allineate tra categorie; si intravedono differenze deboli: **luxury&relax** ha una quota **2★** leggermente superiore, **sport&adventure** evidenzia **3★** un po’ più ampia, **spa** suggerisce una lieve crescita delle **5★**, **culture** sostanzialmente in linea. Nei conteggi, prevale **luxury&relax**, seguono **sport&adventure**, **spa** e infine **culture**.

#### Market
La composizione per stelle è stabile tra i mercati, con scarti contenuti: **at** mostra una quota leggermente più alta di **3★–4★** e un po’ meno **1★**; **uk** presenta un filo più di **1★–2★**; **it** è intermedio. Nei conteggi, i volumi sono sbilanciati: **at** è il mercato più numeroso, seguito da **uk**, poi **it**.

#### Language
Anche qui le percentuali sono vicine: **en** ha una presenza di **1★** leggermente più visibile, mentre **fr** e **it** mostrano fasce **4★–5★** un po’ più ampie. Per volumi, l’ordine è **it > en > fr**.

#### Platform
Le percentuali delle stelle sono molto simili tra **mobile**, **pc** e **tv**; al più si nota su **pc** una lieve quota di **5★** più ampia e un filo meno **2★**, mentre **mobile** e **tv** risultano quasi sovrapponibili. Nei volumi, il traffico è chiaramente diverso: **tv > pc > mobile**.

-----

### 4.2.3 Analisi Bivariate Stelle Vs Variabili

In [ ]:
# === Stars summaries with 95% CI (normal approx) ===
from __future__ import annotations
import numpy as np
import pandas as pd

def stars_group_summary(df: pd.DataFrame, by: str, decimals: int = 3) -> pd.DataFrame:
    """
    Sintesi di 'stars' per gruppi.

    Parametri
    ---------
    df : pandas.DataFrame
        Dataset che contiene la colonna 'stars' (valori 1–5).
    by : str
        Nome della colonna per il raggruppamento
        (es. 'pacchetto_id', 'platform', 'market', 'category', 'destination_id').
    decimals : int, default 3
        Decimali da usare nella formattazione dell’output.

    Ritorna
    -------
    pandas.DataFrame
        Tabella ordinata per media decrescente, poi n decrescente, con:
        - <by>        : valore del gruppo
        - n           : numero di osservazioni nel gruppo
        - share_%     : quota sul totale del dataset
        - mean        : media di 'stars' nel gruppo
        - std         : deviazione standard campionaria
        - se          : errore standard della media (std / sqrt(n))
        - ci95_low    : limite inferiore IC 95% (z = 1.96)
        - ci95_high   : limite superiore IC 95%
        - mean ± CI95 : stringa pronta per i commenti, "m [low, high]"

    Note
    ----
    Le CI usano approssimazione normale; con gruppi piccoli interpretare con
    cautela. Se esistono più visite per lo stesso utente/hotel, l’SE può essere
    ottimistico per via di dipendenze intra-cluster.
    """
    if by not in df.columns:
        raise KeyError(f"Colonna '{by}' non presente nel DataFrame.")

    grp = df.groupby(by, dropna=False)["stars"]
    out = grp.agg(n="count", mean="mean", std="std").reset_index()

    # Evita divisioni per zero
    out["se"] = out["std"] / np.sqrt(out["n"].clip(lower=1))

    z = 1.96  # approx normale; per gruppi molto piccoli interpretare con cautela
    out["ci95_low"]  = out["mean"] - z * out["se"]
    out["ci95_high"] = out["mean"] + z * out["se"]
    out["share_%"]   = out["n"] / len(df) * 100.0

    # Arrotondamenti e colonna "mean ± CI95" per uso diretto nel testo
    num_cols = ["mean", "std", "se", "ci95_low", "ci95_high", "share_%"]
    out[num_cols] = out[num_cols].round(decimals)
    out["mean ± CI95"] = out.apply(
        lambda r: f"{r['mean']:.{decimals}f} [{r['ci95_low']:.{decimals}f}, {r['ci95_high']:.{decimals}f}]",
        axis=1
    )

    # Riordino colonne per leggibilità
    cols = [by, "n", "share_%", "mean", "ci95_low", "ci95_high", "std", "se", "mean ± CI95"]
    out = out[cols].sort_values(["mean", "n"], ascending=[False, False]).reset_index(drop=True)
    return out

# --- Tabelle principali (usa i nomi aggiornati) ---
tbl_pkg   = stars_group_summary(df_base, "pacchetto_id")
tbl_plat  = stars_group_summary(df_base, "platform")
tbl_mkt   = stars_group_summary(df_base, "country")          # ⇒ mercato/lingua
tbl_cat   = stars_group_summary(df_base, "category")
tbl_dest  = stars_group_summary(df_base, "country_id")  #  ⇒ destinazione

display(tbl_pkg)
display(tbl_plat)
display(tbl_mkt)
display(tbl_cat)
display(tbl_dest)

# Esporta in un file Excel per allegati/report
# with pd.ExcelWriter("stars_group_summaries.xlsx", engine="xlsxwriter") as wr:
#     tbl_pkg.to_excel(wr,  index=False, sheet_name="pacchetto")
#     tbl_plat.to_excel(wr, index=False, sheet_name="piattaforma")
#     tbl_mkt.to_excel(wr,  index=False, sheet_name="mercato")
#     tbl_cat.to_excel(wr,  index=False, sheet_name="categoria")
#     tbl_dest.to_excel(wr, index=False, sheet_name="destinazione")

#### Cosa mostra la tabella
Per ogni gruppo sono riportati: numero di casi (*n*), quota sul totale (*share_%*), **media delle stelle**, **intervallo di confidenza al 95%** (IC95%: *low–high*), deviazione standard (*std*) ed errore standard (*se*).
Un IC **stretto** indica una stima più precisa; un IC **ampio** segnala più incertezza. Messaggio chiave: **quasi tutte le medie stanno attorno a 3** e gli IC **si sovrappongono spesso** → differenze piccole, raramente decisive.

**pacchetto_id**
Medie: **0 ≈ 3,23** [3,06–3,41], **1 ≈ 3,02** [2,88–3,15], **2 ≈ 2,99** [2,83–3,15], **3 ≈ 2,96** [2,78–3,13].
Scarto massimo ≈ **0,27 punti** (0 vs 3). IC sovrapposti → **nessuna separazione netta** al 95%.

**platform**
Medie: **tv ≈ 3,31** [3,21–3,41] > **pc ≈ 3,13** (IC più stretto) > **mobile ≈ 3,04** [2,85–3,23].
La differenza **tv–mobile** è la più visibile, ma gli IC **quasi si toccano** → **evidenza debole**.

**market**
Medie: **fr ≈ 3,11** [2,90–3,32], **it ≈ 3,10** [2,98–3,23], **uk ≈ 2,96** [2,81–3,11].
**UK leggermente più basso**, con **ampia sovrapposizione** degli IC.

**category**
Medie: **spa ≈ 3,21** [3,02–3,40], **sport&adventure ≈ 3,11** [2,94–3,28], **culture ≈ 2,98** [2,78–3,21], **luxury&relax ≈ 2,96** [2,82–3,10].
**Spa** è in testa e **luxury&relax/culture** chiudono, ma gli IC **si sovrappongono** → **effetto piccolo**.

**destination_id**
Medie tra **~2,70 e ~3,40** con IC95% **ampi** (campioni ~40–120 per destinazione).
Ci sono destinazioni apparentemente migliori/peggiori, ma gli IC **si sovrappongono in molte coppie**: evitare classifiche “assolute” senza test formali.

#### Lettura complessiva
Le differenze tra gruppi **ci sono ma sono modeste** (tipicamente **≤ 0,3 stelle**). La forte sovrapposizione degli IC suggerisce che molte **potrebbero non essere significative** a α=0,05.
Operativamente, queste variabili **spiegano poco la qualità media**; sono più utili per comprendere **mix** e **volumi** che per **predire** il voto.

-----

## 4.3 Serie Temporali

In [ ]:
def ts_aggregate(
    df: pd.DataFrame,
    freq: str = "W",                 # "D", "W", "M"
    min_count: int = 0,              # riempi date mancanti con 0 per visite
) -> pd.DataFrame:
    """
    Ritorna serie temporale con metriche base:
      - visits: #visite
      - users:  #utenti unici
      - avg_stars: media stelle
    """
    # Normalizza il settimanale a W-MON per coerenza
    freq_norm = "W-MON" if freq == "W" else freq

    g = (df
         .set_index("visit_date")
         .assign(_user=df["user_uuid"])
         .resample(freq_norm))

    out = pd.DataFrame({
        "visits": g.size(),
        "users": g["_user"].nunique(),
        "avg_stars": g["stars"].mean()
    })

    # Calendario completo (zero-fill solo per conteggi)
    if len(out) > 0:
        idx = pd.date_range(out.index.min(), out.index.max(), freq=freq_norm)
        out = out.reindex(idx)
        if min_count is not None:
            out["visits"] = out["visits"].fillna(0).astype("int64")
            out["users"]  = out["users"].fillna(0).astype("int64")
        # avg_stars resta NaN nelle settimane senza visite (corretto)
    return out


def add_rolling(ts: pd.DataFrame, windows=(4,)) -> pd.DataFrame:
    """Aggiunge rolling mean sulle colonne principali (se presenti).
    Le finestre sono in 'periodi' della serie (es. con W-MON: settimane)."""
    ts = ts.copy()
    for w in windows:
        if "visits" in ts:
            ts[f"visits_r{{w}}"] = ts["visits"].rolling(w, min_periods=max(1, w//2)).mean()
        if "avg_stars" in ts:
            ts[f"avg_stars_r{{w}}"] = ts["avg_stars"].rolling(w, min_periods=max(1, w//2)).mean()
    return ts


# ===== 1) Linea visite + media stelle (asse secondario) =====
def plot_ts_visits_and_stars(ts: pd.DataFrame, win: Optional[int] = 4, title="Traffico e rating nel tempo"):
    """
    ts: DataFrame indicizzato per data con colonne 'visits' e 'avg_stars'
    win: finestra per media mobile visite (None per disattivare)
    """
    ts = ts.copy()
    if win and "visits" in ts:
        ts[f"visits_r{win}"] = ts["visits"].rolling(win, min_periods=max(1, win//2)).mean()

    fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                        row_heights=[0.6, 0.4], vertical_spacing=0.06)

    # PANEL 1: visite (bar) + MA opzionale
    fig.add_bar(x=ts.index, y=ts["visits"], name="Visite", row=1, col=1)
    if win and f"visits_r{win}" in ts:
        fig.add_scatter(x=ts.index, y=ts[f"visits_r{win}"], mode="lines",
                        name=f"Visite (MA {win})", row=1, col=1)

    # PANEL 2: media stelle (linea)
    if "avg_stars" in ts:
        fig.add_scatter(x=ts.index, y=ts["avg_stars"], mode="lines",
                        name="Media stelle", row=2, col=1)
        fig.update_yaxes(title_text="Media stelle", range=[1, 5], row=2, col=1)

    fig.update_yaxes(title_text="Visite", row=1, col=1)
    fig.update_xaxes(title_text="Data", row=2, col=1, rangeslider_visible=True)
    fig.update_layout(title=title, hovermode="x unified", margin=dict(l=10, r=10, t=50, b=10))
    fig.show()

### 4.3.1 Andamento delle Valutazioni nel tempo

In [ ]:
# 1) Serie base + rolling mean a 4 periodi (equivale a circa un mese di osservazioni)
ts = ts_aggregate(df_base, freq="W")        # verrà normalizzato a W-MON
ts = add_rolling(ts, windows=(4,))     # MA4 settimanale
plot_ts_visits_and_stars(ts, win=4, title="Traffico e rating nel tempo")

**Traffico.**

Il numero di visite cresce in modo progressivo dal 2021, con un chiaro salto di scala tra fine 2022 e inizio 2023. La media mobile a 4 periodi (MA4) sale stabilmente sopra il livello precedente, raggiunge i massimi nella prima metà del 2023 e poi si assesta leggermente più in basso verso fine anno. La dinamica è quindi di **forte aumento dei volumi** e successiva **stabilizzazione**.

**Valutazioni (media stelle).**

La media delle stelle è volatile quando i volumi sono bassi (2021), poi si **stabilizza intorno a ~3–3,2** con l’aumento del traffico (2022–2023). Non si osserva un trend monotono di miglioramento o peggioramento: le oscillazioni restano contenute entro pochi decimi di stella.

**Relazione tra traffico e rating.**

I picchi di visite non si accompagnano a variazioni persistenti della media stelle. In altre parole, l’incremento del traffico ha modificato soprattutto i **volumi**, non la **qualità media** delle valutazioni.

**Conclusione.**

Nel periodo analizzato le valutazioni **rimangono stabili** attorno a 3 stelle; la principale dinamica temporale riguarda l’**espansione dei volumi** e la successiva normalizzazione. Per il monitoraggio continuo è sufficiente tenere sotto controllo la **% di 1★ e di 4–5★** su finestra mobile (es. MA4/MA8) e aprire un alert solo in caso di scostamenti prolungati rispetto alla media recente.

-----

### 4.3.2 Andamento temporale delle Visite per Stelle

In [ ]:
# ===== 2) Stacked area per STARS (assoluti e 100%) =====
def plot_ts_stars_stacked(
    df: pd.DataFrame,
    freq: str = "W",
    percent: bool = False,
    title: Optional[str] = None,
):
    """
    Stacked area delle visite per classe 'stars' nel tempo (assoluti o %).
    Migliorie:
      - freq settimanale coerente (W-MON)
      - calendario completo (zero-fill) per evitare buchi nei grafici
    """
    # 0) safety: datetime e normalizzazione frequenza
    if not np.issubdtype(df["visit_date"].dtype, np.datetime64):
        df = df.copy()
        df["visit_date"] = pd.to_datetime(df["visit_date"], errors="coerce")
    freq_norm = "W-MON" if freq == "W" else freq

    # 1) prepara le stelle come stringhe ordinate (1→5)
    order_all = ["1", "2", "3", "4", "5"]
    d2 = df.dropna(subset=["stars"]).copy()
    if d2.empty:
        print("[skip] Tutte le stelle sono NaN.")
        return
    d2["stars"] = d2["stars"].astype("Int64")
    d2["stars_str"] = d2["stars"].astype(str)
    # categoria ordinata per avere colori/ordine stabili
    d2["stars_str"] = pd.Categorical(d2["stars_str"], categories=order_all, ordered=True)

    # 2) pivot wide settimanale con calendario completo
    wide = (d2.set_index("visit_date")["stars_str"]
              .groupby(pd.Grouper(freq=freq_norm))
              .value_counts()
              .unstack(level=1))
    if wide is None or wide.empty:
        print("[skip] Nessun dato per 'stars' o 'visit_date'.")
        return

    # assicura tutte le colonne 1..5 (se assenti, aggiungi colonna a 0)
    for c in order_all:
        if c not in wide.columns:
            wide[c] = 0
    wide = wide[order_all]

    # reindicizza su calendario completo
    idx = pd.date_range(wide.index.min(), wide.index.max(), freq=freq_norm)
    wide = wide.reindex(idx).fillna(0).astype("int64")
    wide.index.name = "visit_date"

    # 3) percentuali opzionali
    if percent:
        denom = wide.sum(axis=1).replace(0, np.nan)
        ywide = (wide.div(denom, axis=0) * 100.0).fillna(0.0)
        ytitle = "Percentuale (%)"
    else:
        ywide = wide
        ytitle = "Conteggi"

    # 4) long format per Plotly e grafico
    g = ywide.reset_index().melt(id_vars="visit_date", var_name="stars_str", value_name="y")

    fig = px.area(
        g, x="visit_date", y="y", color="stars_str",
        category_orders={"stars_str": order_all},
        title=title or f"Visite per stelle — {'100% stacked' if percent else 'assoluti'}",
    )
    fig.update_traces(line=dict(width=0.5))
    fig.update_yaxes(title=ytitle, rangemode="tozero", range=[0, 100] if percent else None)
    fig.update_xaxes(title="Data", rangeslider_visible=True)
    fig.update_layout(legend_title_text="Stelle", hovermode="x unified",
                      margin=dict(l=10, r=10, t=50, b=10))
    fig.show()

In [ ]:
plot_ts_stars_stacked(df_base, freq="W-MON", percent=False, title="Visite per stelle (assoluti)")
plot_ts_stars_stacked(df_base, freq="W-MON", percent=True,  title="Visite per stelle (100%)")

**Cosa emerge dalle Percentuali.**

* Nel **2021** le quote cambiano in modo brusco: è **rumore da bassi volumi**.
* Da **metà 2022** in avanti il mix si **stabilizza**: le fasce oscillano ma senza un trend unidirezionale.
* La **somma di 4★ e 5★** rimane in genere intorno a **~40–50%**; **1★–2★** si muovono nell’ordine di **~30–40%**, **3★** completa il resto.
* Negli **ultimi mesi del 2023** si intravede qualche **episodio con quota 1★ più alta**, ma non persistente.

**Conteggi assoluti.**

* I volumi crescono in modo marcato tra **fine 2022 e 2023**.
* L’aumento riguarda **tutte le classi di stelle**: non si osserva una categoria che “prende il sopravvento”; le serie crescono **in modo grosso modo proporzionale**.
* I picchi di traffico (metà 2023) **non cambiano in modo strutturale** il mix: riflettono soprattutto più visite.

In conclusione:

* La **qualità relativa** (percentuale delle classi) è **abbastanza stabile**; le variazioni più nette sono nei **volumi**.
* Tenere un **monitoraggio rolling** della **% 1★** e della **% 4–5★** (es. medie mobili settimanali/mensili) per cogliere deviazioni persistenti; gli sbalzi del 2021 non vanno interpretati senza considerare i **campioni piccoli**.

-----

### 4.3.3 Andamento della Distribuzione delle Variabili nel Tempo

In [ ]:
# ===== 3) Stacked area per CATEGORIA (top_k + 'Altro') =====
def _apply_topk_other(s: pd.Series, top_k: Optional[int], min_share: Optional[float], other_label: str = "Altro"):
    """
    Mantiene i Top-K (per frequenza globale) e/o filtra sotto soglia di quota complessiva.
    Tutto il resto va in 'Altro'. Semplice e deterministico.
    """
    s = s.astype(str).copy()
    vc = s.value_counts()
    keep = set(vc.index)

    if top_k is not None and top_k > 0 and len(keep) > top_k:
        keep = set(vc.nlargest(top_k).index)

    if min_share is not None and 0 < min_share < 1:
        shares = vc / vc.sum()
        # rimuove dai keep chi sta sotto soglia
        keep = keep - set(shares[shares < min_share].index)

    if len(keep) < vc.size:
        s = s.where(s.isin(list(keep)), other_label)

    return s


def plot_ts_stacked_by(
    df: pd.DataFrame,
    by: str = "category",        # usa 'market' o 'destination_id' dopo il rename
    freq: str = "W",
    top_k: Optional[int] = 10,
    min_share: Optional[float] = None,
    percent: bool = False,
    title: Optional[str] = None,
):
    """
    Stacked area di visite nel tempo per il gruppo 'by', con Top-K + 'Altro'.
    Migliorie:
      - 'W' normalizzato a 'W-MON' per settimane complete
      - calendario completo con zero-fill → niente buchi nel grafico
      - ordinamento stabile delle categorie (frequenza globale; 'Altro' in coda)
    """
    if by not in df.columns:
        raise KeyError(f"'{by}' non presente nel DataFrame. (Dopo il rename usa 'market' o 'destination_id')")

    # safety: datetime e freq normalizzata
    if not np.issubdtype(df["visit_date"].dtype, np.datetime64):
        df = df.copy()
        df["visit_date"] = pd.to_datetime(df["visit_date"], errors="coerce")
    freq_norm = "W-MON" if freq == "W" else freq

    # Top-K + Altro su TUTTO il periodo (stabile)
    s = _apply_topk_other(df[by], top_k=top_k, min_share=min_share, other_label="Altro")
    df2 = df.assign(_by=s)

    # Aggregazione → wide (righe=periodi, colonne=gruppi)
    wide = (df2.set_index("visit_date")
              .groupby([pd.Grouper(freq=freq_norm), "_by"])
              .size()
              .unstack("_by"))

    if wide is None or wide.empty:
        print("[skip] Nessun dato dopo l'aggregazione.")
        return

    # Reindex su calendario completo e zero-fill
    idx = pd.date_range(wide.index.min(), wide.index.max(), freq=freq_norm)
    wide = wide.reindex(idx).fillna(0).astype("int64")
    wide.index.name = "visit_date"

    # Ordine categorie: per frequenza globale, 'Altro' in coda
    totals = wide.sum(axis=0).sort_values(ascending=False)
    levels = [c for c in totals.index if c != "Altro"] + (["Altro"] if "Altro" in totals.index else [])
    wide = wide.reindex(columns=levels)

    # Percentuali opzionali
    if percent:
        denom = wide.sum(axis=1).replace(0, np.nan)
        ywide = (wide.div(denom, axis=0) * 100.0).fillna(0.0)
        ytitle = "Percentuale (%)"
    else:
        ywide = wide
        ytitle = "Conteggi"

    # Long per Plotly
    tmp = ywide.reset_index().melt(id_vars="visit_date", var_name="_by", value_name=("perc" if percent else "count"))
    ycol = "perc" if percent else "count"

    fig = px.area(
        tmp, x="visit_date", y=ycol, color="_by",
        category_orders={"_by": levels},
        title=title or f"{by} nel tempo — {'100% stacked' if percent else 'assoluti'}"
    )
    fig.update_traces(line=dict(width=0.4))
    fig.update_yaxes(title=ytitle, rangemode="tozero", range=[0, 100] if percent else None)
    fig.update_xaxes(title="Data", rangeslider_visible=True)
    fig.update_layout(legend_title_text=by, hovermode="x unified", margin=dict(l=10, r=10, t=50, b=10))
    fig.show()

In [ ]:
plot_ts_stacked_by(df_base, by="category",     freq="W-MON", top_k=8,  percent=False, title="Tema della vacanza (Valori Assoluti)")
plot_ts_stacked_by(df_base, by="category",     freq="W-MON", top_k=8,  percent=True, title="Tema della vacanza (quota %)")

#### Tema della Vacanza

I volumi crescono lungo tutto l’orizzonte e accelerano fra fine 2022 e il 2023. L’aumento riguarda **tutte le categorie**, ma il contributo maggiore viene da **luxury&relax**, che resta la fetta più ampia in quasi tutto il periodo. **sport&adventure** è il secondo driver, mentre **spa** e **culture** pesano meno ma seguono lo stesso profilo di crescita.

* Nel **2021** le percentuali oscillano molto: è l’effetto dei **bassi volumi**.
* Dal **2022** il mix si **stabilizza**:

  * **luxury&relax** mantiene la **quota prevalente** (spesso intorno a ~45–60%).
  * **sport&adventure** si colloca generalmente **al secondo posto** (~20–35%).
  * **spa** rimane **terza** (~10–20%), con qualche fase di lieve risalita.
  * **culture** è **la più piccola** (tipicamente <~10–15%), con sporadici picchi.

* La crescita nel tempo è **trainata dal volume**, non da un cambio strutturale di preferenze: il **mix fra categorie resta simile** quando i numeri diventano robusti (dal 2022).
* **luxury&relax** è la linea guida per la **pianificazione** (domanda più alta e stabile); **sport&adventure** assicura un secondo canale importante; **spa** e **culture** hanno nicchie più piccole ma costanti.

**Monitoraggio consigliato:**

* Tenere sotto controllo, per mese, le **quote %** delle quattro categorie e un indice di **concentrazione** (es. HHI) per intercettare eventuali spostamenti di preferenza.
* Quando i volumi cambiano molto, distinguere sempre **effetto volume** da **effetto mix** (grafici assoluti vs. 100%).

In [ ]:
plot_ts_stacked_by(df_base, by="platform", freq="W-MON", top_k=5, percent=False, title="Piattaforma Utilizzata (Valori Assoluti)")
plot_ts_stacked_by(df_base, by="platform", freq="W-MON", top_k=5, percent=False, title="Piattaforma Utilizzata (quota %)")

#### Piattaforma Utilizzata

I volumi crescono su tutto l’orizzonte e accelerano tra fine **2022** e **2023**. L’aumento riguarda **tutte** le piattaforme, con ordine stabile: **tv** è la più usata, segue **pc**, poi **mobile**.

* Nel **2021** le quote sono molto volatili per via dei **bassi volumi**.
* Dal **2022** il mix si **stabilizza**: **tv** mantiene la **quota maggioritaria** (circa metà o più del totale), **pc** resta seconda e **mobile** rimane la quota minore.
* Si vedono oscillazioni di breve periodo (alcune settimane con più **pc** o **mobile**), ma **nessun cambio strutturale** dell’ordine.

* La crescita recente è **trainata soprattutto da tv**, ma anche **pc** e **mobile** aumentano coerentemente con i volumi totali.
* Per la pianificazione: priorità a **tv** (capacità e contenuti), con **pc** come secondo canale; **mobile** è più piccolo ma in crescita e va mantenuto efficiente.

**Monitoraggio consigliato:**

* Tracciare mensilmente le **quote %** per piattaforma e confrontarle con i **volumi assoluti** per distinguere effetti di **mix** da effetti di **traffico**.
* Definire soglie di attenzione se la quota **tv** scende o **mobile** supera stabilmente una soglia (es. +5–10 pt) rispetto alla media semestrale.

In [ ]:
plot_ts_stacked_by(df_base, by="pacchetto_id", freq="W-MON", top_k=None, percent=False, title="Pacchetto ID (Valori Assoluti)")
plot_ts_stacked_by(df_base, by="pacchetto_id", freq="W-MON", top_k=None, percent=True, title="Pacchetto ID (quota %)")

#### Pacchetto ID

I volumi aumentano in modo marcato da fine **2022** e per tutto il **2023**. La crescita coinvolge tutti i pacchetti; nel complesso **il pacchetto 2** contribuisce più spesso alla parte maggiore del traffico, mentre **0** e **3** mostrano **picchi episodici** (soprattutto nel 2023). **1** è più regolare e costante.

* Nel **2021** le quote saltano molto: è rumore da **bassi volumi**.
* Dal **2022** il mix si **stabilizza** senza un vincitore assoluto:

  * **2** tende a essere **il più frequente**,
  * **1** segue a distanza ridotta,
  * **0** e **3** oscillano e alternano fasi di risalita, specie nel 2023.

* La dinamica principale è **l’aumento dei volumi**, non uno spostamento radicale tra pacchetti.
* Per la pianificazione conviene dimensionare capacità e contenuti partendo da **pacchetto 2** (driver più stabile), mantenendo **1** come secondo pilastro e monitorando **0** e **3** per gestire i loro picchi.

**Monitoraggio consigliato:**

* Report **mensile** con quote % per pacchetto + volumi assoluti (per distinguere **effetto mix** da **effetto traffico**).
* Alert se la quota di **2** scende stabilmente o se **0/3** superano la loro media semestrale di **alcuni punti** per più settimane consecutive.

In [ ]:
plot_ts_stacked_by(df_base, by="country_id",  freq="W-MON", top_k=12, percent=False,  title="Destinazioni (Valori Assoluti)")
plot_ts_stacked_by(df_base, by="country_id",  freq="W-MON", top_k=12, percent=True,  title="Destinazioni (quota %)")

#### Destinazioni

I volumi crescono in modo continuo dal 2021 e accelerano tra fine **2022** e **2023**. La crescita riguarda tutte le destinazioni, ma **una in particolare** (la serie che occupa la porzione più ampia dell’area impilata) diventa il **motore principale** dei volumi nel 2023. Le altre destinazioni aumentano, ma restano **più frammentate** e con picchi episodici.

* Nel **2021** le quote sono molto instabili: è l’effetto dei **bassi numeri**.
* Dal **2022** il mix si **concentra**: la destinazione dominante mantiene **la quota più alta** per gran parte del tempo, mentre le restanti si alternano senza un **rimpasto strutturale**.
* Anche nel **2023** la **concentrazione rimane elevata**: il leader resta tale; si vedono oscillazioni di breve periodo tra le altre destinazioni ma nessun sorpasso duraturo.

Il traffico cresce soprattutto grazie a **una destinazione leader** e a una **coda lunga** di mete minori. Per la pianificazione conviene:

* dimensionare capacità e contenuti partendo dalla destinazione dominante;
* mantenere offerte “snelle” sulle mete minori, gestendo i loro **picchi intermittenti**.

**Monitoraggio consigliato:**

* Report mensile di **Top-1 share** e **Top-3 share** per misurare la concentrazione; aggiungere un indice (es. **HHI**) per capire se il portafoglio si sta polarizzando o diversificando.
* Alert se la quota del leader scende stabilmente o se una destinazione minore **raddoppia** la propria quota rispetto alla media semestrale per più settimane: possibile cambio di preferenze o campagna locale da sfruttare/indagare.

In [ ]:
plot_ts_stacked_by(df_base, by="hotel_id",    freq="W-MON", top_k=15, percent=False, title="Top hotel nel tempo (Valori Assoluti)")
plot_ts_stacked_by(df_base, by="hotel_id",    freq="W-MON", top_k=15, percent=True, title="Top hotel nel tempo (quota %)")

#### Hotel ID

**Cosa vediamo nel grafico**

Il mix per **hotel\_id** è **estremamente frammentato**: le linee dei singoli hotel sono sottili e intermittenti, mentre la fascia **“Altro”** domina gran parte dell’area. Questo riflette un catalogo molto ampio con **tante strutture a bassa frequenza** e pochissime ricorrenze significative.

**Dati chiave**

* **Top-10 hotel\_id**: **\~2,4%** del totale
* **Coda lunga (“Altro”)**: **\~97,6%**
* **Totale visite**: **999** — **Finestra**: **2021–2023**

**Cosa tenere a mente**

* La lettura **assoluta** può risultare “rumorosa” per via dei conteggi minimi per singolo hotel; la vista **%** aiuta a stabilizzare.
* Ha senso usare **Top-K + “Altro”** e, se disponibile, **cluster/macro-aree** per una lettura più strategica.
* Gli **hotel\_id** sono identificativi: senza una mappatura a nomi/aree, i commenti restano **quantitativi**.

In [ ]:
plot_ts_stacked_by(df_base, by="language",    freq="W-MON", top_k=8,  percent=False,  title="Lingua utilizzata nell'app (Valori Assoluti)")
plot_ts_stacked_by(df_base, by="language",    freq="W-MON", top_k=8,  percent=True,  title="Lingua utilizzata nell'app (quota %)")

#### Lingua utilizzata nell’app

I volumi crescono in modo consistente dal 2021, con un’accelerazione tra fine **2022** e **2023**. L’aumento riguarda tutte le lingue, ma la fascia **it** resta la più corposa; **en** segue con una crescita visibile; **fr** mantiene numeri più contenuti con qualche picco.

* Nel **2021** le percentuali sono molto instabili (bassi volumi).
* Dal **2022** il mix si **stabilizza**:

  * **it** mantiene la **quota maggioritaria** (tipicamente ~50–70%).
  * **en** si assesta come **seconda lingua** (~25–40%), con oscillazioni di periodo.
  * **fr** rimane **minoritaria** (<~10–20%) con sporadiche risalite, senza cambi di regime duraturi.

* La crescita complessiva è trainata soprattutto da **it**, ma anche **en** contribuisce in maniera costante.
* Le preferenze linguistiche **non cambiano struttura**: utile per pianificare contenuti/assistenza con priorità **it → en → fr**.

**Monitoraggio consigliato:**

* Cruscotto mensile con **quote %** per lingua accanto ai **volumi assoluti**, per separare effetto **mix** da **traffico**.
* Alert se **en** o **fr** aumentano di **+5–10 punti** in modo persistente (≥ 4 settimane), segnale di spostamento di audience o campagne locali da approfondire.

In [ ]:
plot_ts_stacked_by(df_base, by="country",     freq="W-MON", top_k=12, percent=False,  title="Mercato utenti (Valori Assoluti)")
plot_ts_stacked_by(df_base, by="country",     freq="W-MON", top_k=12, percent=True,  title="Mercato utenti (quota %)")

#### Mercato utenti

I volumi crescono lungo tutto l’orizzonte e accelerano tra fine **2022** e **2023**. L’incremento riguarda tutti i mercati, ma l’apporto principale è di **it**; **uk** segue con andamento regolare; **fr** resta più contenuto con qualche picco.

* Nel **2021** le quote sono molto instabili (bassi numeri).
* Dal **2022** il mix si **stabilizza**:

  * **it** mantiene una **quota maggioritaria** (≈ **50–70%**).
  * **uk** si colloca **secondo** (≈ **25–40%**), con oscillazioni ma senza sorpassi duraturi.
  * **fr** rimane **minoranza** (≈ **5–15%**), con risalite episodiche soprattutto nel 2023.
    Non si osservano cambi di regime: l’ordine **it → uk → fr** resta invariato.

La crescita è trainata da **it**, con **uk** come secondo pilastro stabile. Le preferenze per mercato non cambiano struttura, quindi la pianificazione commerciale può seguire le stesse priorità: copertura forte su **it**, presidio costante su **uk**, approccio più mirato su **fr**.

-----

### 4.3.4 Maturità Utenti

In [ ]:
bins   = [-float("inf"), 7, 14, 30, 60, 90, 180, 365, 730, float("inf")]
labels = ["≤7g", "8–14g", "15–30g", "31–60g", "61–90g", "91–180g", "181–365g", "366-730g", ">730g"]
df_base["age_bucket"] = pd.cut(df_base["user_maturity_days"], bins=bins, labels=labels)

plot_ts_stacked_by(df_base, by="age_bucket", freq="W-MON", top_k=None, percent=False, title="Maturità utenti (Valori Assoluti per bucket)")
plot_ts_stacked_by(df_base, by="age_bucket", freq="W-MON", top_k=None, percent=True, title="Maturità utenti (quota % per bucket)")

#### Maturità utenti

I volumi crescono lungo tutto l’orizzonte e accelerano tra fine 2022 e il 2023. La crescita è trainata quasi interamente dai bucket **più maturi**: prima **181–365 giorni**, poi soprattutto **>365 giorni**, che diventano la parte più consistente del traffico. I bucket più giovani (<90 giorni) restano marginali e intermittenti.

* Nel **2021** le percentuali saltano per via dei **bassi volumi**.
* Dal **2022** il mix si stabilizza e si **sposta verso l’alto**: i bucket **>730g** e **366-730g** diventano **prevalenti** (spesso >50%, talvolta molto di più), **181–365g** è il terzo contributo, mentre **<180g** pesa poco e senza trend di crescita duraturo.

Questo profilo è coerente con un **parco utenti che invecchia**: le coorti acquisite in passato continuano a usare l’app (buona **retention**), mentre l’**ingresso di utenti nuovi** è più contenuto rispetto al peso dei “veterani”. 

È anche un effetto **meccanico di maturazione delle coorti**: col passare del tempo, gli utenti “scivolano” naturalmente nei bucket più alti. Per capire se l’acquisizione stia rallentando davvero, va separato l’effetto di **crescita dei volumi** dall’effetto **età/coorte**.

-----

### 4.3.5 Heatmap delle Visite 

In [ ]:
# ===== 4) Heatmap traffico: settimana x giorno =====
def plot_weekly_dow_heatmap(df: pd.DataFrame, title="Heatmap visite (settimana × giorno)"):
    """
    Mostra una heatmap per ciascun anno ISO: righe = giorno (L→D), colonne = settimana (1→53), celle = #visite.
    Migliorie:
      - griglia completa settimana×giorno con zero-fill (niente buchi visivi)
      - scala colore coerente tra anni (zmax globale)
      - ordinamento stabile (Lun→Dom, week 1→53)
    """
    d = df.copy()

    # Safety: assicura datetime
    if not np.issubdtype(d["visit_date"].dtype, np.datetime64):
        d["visit_date"] = pd.to_datetime(d["visit_date"], errors="coerce")

    # Estrazione componenti ISO (pandas 2.x: UInt32 → cast a int)
    iso = d["visit_date"].dt.isocalendar()
    d["year"] = iso.year.astype(int)
    d["week"] = iso.week.astype(int)
    d["dow"]  = d["visit_date"].dt.dayofweek.astype(int)  # 0=Mon ... 6=Sun

    # Aggrega conteggi
    mat = (d.groupby(["year", "week", "dow"])
             .size()
             .rename("visits")
             .reset_index())

    if mat.empty:
        print("[skip] Nessun dato per heatmap settimana×giorno.")
        return

    # Etichette ordinate L→D
    dow_names = ["Lun", "Mar", "Mer", "Gio", "Ven", "Sab", "Dom"]
    mat["dow_name"] = mat["dow"].map(dict(enumerate(dow_names)))

    # zmax globale per comparabilità tra anni
    zmax_global = int(mat["visits"].max())

    # Una heatmap per anno
    for yr, g in mat.groupby("year"):
        # Pivot e reindex coerente su tutta la griglia (righe L→D, colonne 1..53)
        pivot = (g.pivot(index="dow_name", columns="week", values="visits")
                   .reindex(index=dow_names))
        pivot = pivot.reindex(columns=range(1, 54)).fillna(0).astype("int64")

        # Disegno
        fig = px.imshow(
            pivot,
            aspect="auto",
            color_continuous_scale="Viridis",
            zmin=0, zmax=zmax_global,
            title=f"{title} — {yr}"
        )
        fig.update_xaxes(title="Settimana ISO", tickmode="auto")
        fig.update_yaxes(title="")
        fig.update_layout(coloraxis_colorbar=dict(title="Visite"),
                          margin=dict(l=10, r=10, t=50, b=10))
        fig.show()


In [ ]:
plot_weekly_dow_heatmap(df_base)

####  Analisi delle Heatmap

* **2020:** praticamente **assenza di attività**, salvo un episodio isolato a fine anno.
* **2021:** attività **rara e discontinua** (pochi slot accesi, intensità bassa).
* **2022:** più **giorni attivi** per settimana, ma ancora a intensità contenuta.
* **2023:** la griglia è **densa e uniforme**: molte settimane con più giorni attivi e punte fino a ~5–6 visite/giorno.

Non emerge una preferenza netta per un singolo giorno: nel 2023 tutte le righe (lun–dom) risultano regolarmente attive. Questo conferma quanto visto prima sui grafici per *visit_dow_num*: **l’effetto “giorno” è debole** e le differenze sono soprattutto di **volume complessivo** tra gli anni.

* Il salto vero è **di scala**: da poche “macchie” nel 2021 a una **copertura quasi continua** nel 2023.
* In assenza di un giorno dominante, la **pianificazione** dovrebbe essere distribuita sull’intera settimana, con capacità che segue il **volume totale** più che il giorno specifico.

**Monitoraggio utile**

* **Active days / week** (quanti giorni con ≥1 visita): aspettarsi una crescita netta 2021→2023; usare questo indicatore come segnale di **ampiezza della domanda**.
* **Visite medie per giorno attivo** e **p95** per intercettare settimane “calde”.
* **Anomalie**: mappa z-score su settimana×giorno per evidenziare cluster insolitamente alti/bassi (utile per alert operativi).

Le heatmap mostrano il passaggio da un utilizzo sporadico a una **presenza costante durante tutta la settimana**, senza pattern forti per singolo giorno; l’attenzione va quindi ai **volumi complessivi** e alle **settimane anomale**, più che al weekday in sé.

-----

# 5.0 Conclusioni Finali

Nella mia EDA ho analizzato **999 visite** (11 colonne) su esperienze di viaggio virtuali. Ho verificato schema e qualità dei dati, ho costruito feature temporali e viste percentuali/assolute, e ho affiancato alle letture grafiche indicatori sintetici.

## Cosa ho esplorato e cosa ho trovato

* **Volumi e stagionalità.**
Ho una crescita netta e sbilanciata sul **2023 (60,7%)**, dopo **2022 (28,8%)** e **2021 (10,5%)**. A livello mensile il massimo è **Aprile (≈9,01%)**; le differenze tra mesi restano contenute. Per giorno della settimana il **Giovedì (≈16,1%)** è il picco, ma gli altri giorni sono vicini (Lun 15,2% … Dom 14,8%): non emerge un “giorno re”, il profilo è sostanzialmente equilibrato.

* **Mix per categoria.**
La domanda è polarizzata su **comfort/benessere**: **luxury&relax ~42% (419)**, **sport&adventure ~26% (262)**, **spa ~19% (194)**, **culture ~12% (124)**. È un portafoglio dove il relax domina ma esiste uno zoccolo duro di “attività leggere” e benessere.

* **Valutazioni (stelle).**
La distribuzione è **molto uniforme**: **1★ 19,32% (193)** · **2★ 18,92% (189)** · **3★ 19,92% (199)** · **4★ 21,02% (210)** · **5★ 20,82% (208)**. Tratto le stelle come **ordinale**: quando confronto gruppi considero mediana/IQR oltre alla media. Nel tempo, il **mix è stabile**: la somma **4★+5★** si muove tipicamente **~40–50%**, **1★+2★ ~30–40%**, con il 2023 che cresce soprattutto in **volume** senza cambiare struttura.

* **Piattaforme e soddisfazione.**
Il traffico è ordinato **tv > pc > mobile**. Sulle medie stelle vedo: **tv ≈ 3,31 [3,21–3,41] > pc ≈ 3,13 > mobile ≈ 3,04 [2,85–3,23]**. Gli **intervalli di confidenza si sfiorano** → differenze **piccole**: segnale interessante, non una spaccatura.

* **Lingue/mercati.**
Dal 2022 il mix si **stabilizza**: **it maggioritaria (≈50–70%)**, **en seconda (≈25–40%)**, **fr minoritaria (<≈10–20%)**. Le quote si muovono senza cambi di regime: la pianificazione pratica rimane **it → en → fr**.

* **Maturità utenti.**
I bucket “anziani” dominano: le visite arrivano soprattutto da sottoscrizioni lontane nel tempo, mentre i bucket più giovani (<90/180 giorni) restano marginali. Questa lettura è coerente con un **parco utenti che invecchia**; distinguo sempre l’effetto **crescita** dall’effetto **coorte**.

* **Coda lunga degli hotel.**
È un catalogo frammentato: **Top-10 hotel_id ≈ 2,4%**, **“Altro” ≈ 97,6%**. Per una lettura strategica preferisco **Top-K + “Altro”** o cluster/macro-aree.

* **Heatmap settimana×giorno.**
Nel 2023 la griglia è **densa e uniforme**; non c’è un singolo giorno strutturalmente dominante. Pianifico quindi per **capacità complessiva** più che per giorno “speciale”. Monitor utili: **Active days/week**, **visite medie per giorno attivo** e **p95**, più una mappa di **anomalie (z-score)**.

* **Relazioni tra variabili e stelle.**
Le differenze tra gruppi **ci sono ma sono modeste** e gli **IC 95% spesso si sovrappongono**: utili per comprendere **mix e volumi**, meno per “predire” il voto. Sulle categorie, ad esempio, **spa (~3,21 [3,02–3,40])** e **sport&adventure (~3,11)** risultano lievemente sopra **culture (~2,98 [2,78–3,21])** e **luxury&relax (~2,96 [2,82–3,10])**, ma gli intervalli indicano **effetti piccoli**.

## Come interpreto il quadro

1. L’aumento dei volumi nel 2023 **non** ha deformato la qualità percepita: il mix delle stelle è stabile.
2. **Uso living-room, soddisfazione diffusa.**  **tv** porta molta scala e ha la media più alta, ma le differenze con **pc/mobile** sono **ridotte**; il prodotto si comporta in modo coerente tra canali.
3. Il peso di **luxury&relax** e la presenza stabile di **spa** confermano un bisogno di comfort; **sport&adventure** copre una fascia attiva ma accessibile.
4. Le priorità di lingua/mercato restano stabili (**it → en → fr**), utile per pianificare contenuti e assistenza.
5. Le visite sono trainate da iscritti “anziani”; i bucket giovani sono poco rappresentati.

## Cosa propongo di fare

* **Monitoraggio “a vista” delle stelle.**  Tengo un **rolling** su **% 1★** e **% 5★**, leggendo sempre le variazioni alla luce dei **campioni piccoli**.
* **KPI operativi per capacità.**  Traccio **Active days/week**, **visite medie per giorno attivo** e **p95**; uso una **mappa z-score** per intercettare settimane/giorni anomali.
* **Quote per piattaforma, con soglie.**  Aggiorno mensilmente le **quote % tv/pc/mobile** e definisco **soglie di attenzione** su scostamenti (es. +5–10pt rispetto alla media semestrale).
* **Top-K + “Altro” su hotel_id.**  Standardizzo la vista per attenuare il rumore della coda lunga e per dare leggibilità editoriale.
* **Lettura ordinale delle stelle.**  Nei confronti tra gruppi riporto **mediana e IQR** oltre alle medie con **IC 95%**, evitando classifiche “assolute” quando gli IC si sovrappongono.
* **Priorità di contenuto e assistenza.**  Mantengo la pipeline **it** come priorità, presidio costante su **en**, azioni mirate e proporzionate su **fr**.

## Limiti che assumo esplicitamente

* Il dataset è **sbilanciato sul 2023**: distinguo sempre **crescita reale** da **effetto finestra**.
* Uso **visite** e non **utenti unici** nella maggior parte delle viste: i ricorrenti pesano di più nelle percentuali.
* Le differenze tra gruppi sono **piccole**: gli **IC** suggeriscono di trattarle come **ipotesi operative** da seguire nel monitoraggio, non come verità scolpite.

-----

In [ ]:
df_base.to_csv("Travel_data Final Project Updated.csv")

Ti ringrazio per aver letto il mio notebook.

Visita il mio [blog personale](https://stefanonocco.it/) o colleghiamoci su:  
- [LinkedIn](https://www.linkedin.com/in/stefanonocco/)  
- [Instagram](https://www.instagram.com/steno.py/)  

***Stefano Nocco***